# 🔧 Исправление Canonical формата Income Statement на основе официальной отчетности

**Цель**: Создать правильный canonical формат для истории IS, который соответствует реальным данным из официальной отчетности

**Задача**: 
1. Загрузить данные из официального Income Statement 2024
2. Сравнить с RAW данными из БД (как данные загрузились в БД до маппинга)
3. Сравнить с CANONICAL данными из БД (как данные стали после маппинга в DataMart)
4. Выявить расхождения и понять, правильно ли препроцессинг собрал отчет о прибылях и убытках
5. Создать исправленный canonical формат для 2024 года на основе официальных данных
6. **Обновить CSV файл истории с исправленными данными**
7. **Загрузить исправленные данные в RAW БД через `load_history_csv_to_db`**
8. **Проверить, что маппинг RAW → CANONICAL работает правильно**
9. **Убедиться, что препроцессинг правильно использует canonical данные**

**Процесс загрузки данных**:
```
Официальный отчет → CSV файл истории (RAW формат) → RAW БД → CANONICAL БД (через маппинг) → Препроцессинг → Движок
```

**Примечание**: CSV файлы истории используются DataMart как fallback, но для сравнения мы работаем напрямую с БД (RAW и CANONICAL форматы).

**Важно**: 
- Используйте данные в том же порядке, как в официальном отчете
- Проверьте порядок цифр (могут быть ошибки в масштабе)
- Убедитесь, что все статьи правильно сопоставлены с canonical метриками
- **CSV файл истории должен содержать данные в RAW формате (как в официальном отчете)**
- **Маппинг RAW → CANONICAL происходит автоматически при загрузке через DataMart**

**⭐ Новые метрики breakdown (опциональные)**:
- **Revenue**: `revenue_product`, `revenue_service`, `revenue_other` → `revenue_total`
- **COGS**: `cogs_materials`, `cogs_labor`, `cogs_overhead`, `cogs_services`, `cogs_other` → `cogs_total`
- **SGA**: `sga_selling`, `sga_general_admin`, `sga_marketing`, `sga_it`, `sga_other` → `sga_total`
- **Taxes**: `current_tax_expense`, `deferred_tax_expense` → `total_tax_expense`
- **Net Income**: `net_income_parent`, `net_income_nci` → `net_income`
- **Margins**: `gross_margin`, `operating_margin` (вычисляются автоматически)
- **Interest**: `interest_expense_debt`, `interest_expense_lease` → `interest_expense`

Эти метрики опциональны и используются для более детального моделирования, если доступны в исходных данных.

## 🔧 Импорты и настройка

In [23]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import sys
from IPython.display import display, Markdown, HTML
from typing import Optional
from difflib import SequenceMatcher
import re

# Настройка для графиков в Jupyter
%matplotlib inline

# Определение корня проекта (ПЕРЕД импортом engine)
current_dir = Path.cwd()
if (current_dir / 'engine').exists():
    ROOT = current_dir
elif (current_dir.parent / 'engine').exists():
    ROOT = current_dir.parent
elif (current_dir.parent.parent / 'engine').exists():
    ROOT = current_dir.parent.parent
else:
    nb_path = Path(__file__) if '__file__' in globals() else Path.cwd()
    ROOT = nb_path.parent.parent.parent

print(f"ROOT: {ROOT}")
print(f"Engine exists: {(ROOT / 'engine').exists()}")

# Добавляем ROOT в sys.path ПЕРЕД импортом engine
sys.path.insert(0, str(ROOT))

# Теперь импортируем engine
from engine.database.data_mart import get_data_mart

# Определяем COMPANY автоматически из пути ноутбука
COMPANY = 'us_steel'  # fallback по умолчанию
if 'companies' in current_dir.parts:
    companies_idx = current_dir.parts.index('companies')
    if companies_idx + 1 < len(current_dir.parts):
        COMPANY = current_dir.parts[companies_idx + 1]

croot = ROOT / 'companies' / COMPANY
print(f"Company root: {croot}")

MODEL_VERSION: Optional[str] = None

ROOT: /home
Engine exists: True
Company root: /home/companies/us_steel


## 📋 ШАГ 1: Данные из официального Income Statement 2024

In [24]:
# === Вставьте данные из официального Income Statement 2024 ===
# Данные из официальной отчетности US Steel (в том же порядке, как в отчете)

official_reporting_2024 = {
    # === REVENUE ===
    'Net sales': 13135.0,  # миллионов USD
    'Net sales to related parties': 2505.0,
    'Total net sales': 15640.0,  # Total revenue
    
    # === OPERATING EXPENSES ===
    'Cost of sales (excludes items shown below)': 14060.0,
    'Selling, general and administrative expenses': 435.0,
    'Depreciation, depletion and amortization': 913.0,
    'Earnings from investees': -112.0,  # отрицательное значение = доход
    'Asset impairment charges': 19.0,
    'Restructuring and other charges': 8.0,
    'Gain on equity investee transactions': 0.0,
    'Other losses (gains), net': 77.0,
    'Total operating expenses': 15400.0,
    
    # === OPERATING INCOME ===
    'Earnings before interest and income taxes (EBIT)': 240.0,
    
    # === INTEREST & OTHER FINANCIAL ITEMS ===
    'Interest expense': 24.0,
    'Interest income': -96.0,  # отрицательное значение = доход
    'Loss on debt extinguishment': 2.0,
    'Other financial costs': 29.0,
    'Net periodic benefit income': -132.0,  # отрицательное значение = доход
    'Net gain from investments related to active employee benefits': -25.0,  # отрицательное значение = доход
    'Net interest and other financial (benefits) costs': -198.0,  # отрицательное значение = доход
    
    # === INCOME BEFORE TAXES ===
    'Earnings before income taxes (EBT)': 438.0,
    
    # === TAXES ===
    'Income tax expense': 54.0,
    
    # === NET INCOME ===
    'Net earnings': 384.0,
    'Net earnings attributable to noncontrolling interests': 0.0,
    'Net earnings attributable to United States Steel Corporation': 384.0,
    
    # === EPS ===
    'Earnings per share (basic)': 1.71,  # уже в долларах
    'Earnings per share (diluted)': 1.57,  # уже в долларах
}

# Конвертируем из миллионов в доллары (кроме EPS, которые уже в долларах)
official_reporting_2024_converted = {}
for k, v in official_reporting_2024.items():
    if 'per share' in k.lower() or 'eps' in k.lower():
        # EPS остается как есть (уже в долларах)
        official_reporting_2024_converted[k] = v
    else:
        # Остальные метрики конвертируем из миллионов в доллары
        official_reporting_2024_converted[k] = v * 1_000_000

official_reporting_2024 = official_reporting_2024_converted

print(f"✅ Загружено {len(official_reporting_2024)} статей из официального Income Statement 2024")

# Создаём таблицу для отображения
official_df = pd.DataFrame([
    {
        '№': i + 1,
        'Статья (официальный отчет)': item,
        'Значение (млн USD)': value / 1_000_000 if abs(value) > 1000 else value,
        'Значение (USD)': value if abs(value) > 1000 else f"{value:.2f}",
        'Единица': 'млн USD' if abs(value) > 1000 else 'USD'
    }
    for i, (item, value) in enumerate(official_reporting_2024.items())
])

display(Markdown("### 📊 Таблица данных из официального Income Statement 2024"))
display(official_df.style.format({
    'Значение (млн USD)': lambda x: f"${x:,.2f}" if abs(x) > 0.01 else "$0.00",
    'Значение (USD)': lambda x: f"${x:,.0f}" if isinstance(x, (int, float)) and abs(x) > 1000 else str(x)
}).set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
]).set_properties(**{'font-size': '9px'}))

print(f"\n📋 Всего статей: {len(official_reporting_2024)}")
print(f"📊 Проверка связки: Revenue - Operating Expenses = EBIT")
revenue = official_reporting_2024.get('Total net sales', 0)
opex = official_reporting_2024.get('Total operating expenses', 0)
ebit = official_reporting_2024.get('Earnings before interest and income taxes (EBIT)', 0)
print(f"   Revenue: ${revenue/1_000_000:,.0f}M")
print(f"   Operating Expenses: ${opex/1_000_000:,.0f}M")
print(f"   EBIT: ${ebit/1_000_000:,.0f}M")
print(f"   Проверка: ${revenue/1_000_000:,.0f}M - ${opex/1_000_000:,.0f}M = ${(revenue-opex)/1_000_000:,.0f}M (ожидается ${ebit/1_000_000:,.0f}M)")

✅ Загружено 27 статей из официального Income Statement 2024


### 📊 Таблица данных из официального Income Statement 2024

,№,Статья (официальный отчет),Значение (млн USD),Значение (USD),Единица
0,1,Net sales,"$13,135.00","$13,135,000,000",млн USD
1,2,Net sales to related parties,"$2,505.00","$2,505,000,000",млн USD
2,3,Total net sales,"$15,640.00","$15,640,000,000",млн USD
3,4,Cost of sales (excludes items shown below),"$14,060.00","$14,060,000,000",млн USD
4,5,"Selling, general and administrative expenses",$435.00,"$435,000,000",млн USD
5,6,"Depreciation, depletion and amortization",$913.00,"$913,000,000",млн USD
6,7,Earnings from investees,$-112.00,"$-112,000,000",млн USD
7,8,Asset impairment charges,$19.00,"$19,000,000",млн USD
8,9,Restructuring and other charges,$8.00,"$8,000,000",млн USD
9,10,Gain on equity investee transactions,$0.00,0.00,USD



📋 Всего статей: 27
📊 Проверка связки: Revenue - Operating Expenses = EBIT
   Revenue: $15,640M
   Operating Expenses: $15,400M
   EBIT: $240M
   Проверка: $15,640M - $15,400M = $240M (ожидается $240M)


## 📄 ШАГ 2: Загрузка данных из файла истории (опционально)

**Примечание**: Этот шаг опционален. CSV файл (`is_history_us_steel.csv`) - это результат конвертации Excel файла через `excel_loader.py`. 

**Процесс загрузки данных**:
1. **Excel файл** → `excel_loader.py` → **CSV файл** (без маппинга, как есть)
2. **CSV файл** → `load_history_csv_to_db` → **RAW БД** (загружается как есть)
3. **RAW БД** → `normalize_to_canonical` + `excel_loader.yaml` → **CANONICAL БД** (применяется маппинг)

CSV файлы используются DataMart как fallback, но основная работа идет с данными из БД (ШАГ 3). 

Если хотите проверить, что данные из Excel правильно конвертировались в CSV и загрузились в БД, можете использовать этот шаг. Иначе переходите к ШАГ 3.

In [25]:
# === Загрузка данных из файла истории IS ===

hist_file = croot / "history" / "is_history_us_steel.csv"

if not hist_file.exists():
    print(f"❌ Файл истории не найден: {hist_file}")
    hist_2024_raw = {}
else:
    hist_df = pd.read_csv(hist_file)
    print(f"✅ Файл истории загружен: {hist_file.name}")
    print(f"   Структура: {hist_df.shape[0]} строк, {hist_df.shape[1]} колонок")
    
    # Получаем данные 2024 из файла истории
    hist_2024_raw = {}
    if '2024' in hist_df.columns:
        for _, row in hist_df.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['2024'], errors='coerce')
            if pd.notna(value):
                hist_2024_raw[metric] = float(value)
        
        print(f"\n✅ Загружено {len(hist_2024_raw)} статей из файла истории для 2024 года")
        
        # Создаём таблицу для отображения
        hist_df_display = pd.DataFrame([
            {
                '№': i + 1,
                'Статья (файл истории)': item,
                'Значение (млн USD)': value / 1_000_000,
                'Значение (USD)': value
            }
            for i, (item, value) in enumerate(sorted(hist_2024_raw.items()))
        ])
        
        display(Markdown("### 📄 Таблица данных из файла истории (2024 год)"))
        display(hist_df_display.style.format({
        'Значение (млн USD)': '${:,.2f}',
        'Значение (USD)': '${:,.0f}'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '9px'}))
    else:
        print(f"❌ Колонка '2024' не найдена в файле истории")
        print(f"   Доступные колонки: {list(hist_df.columns)}")

✅ Файл истории загружен: is_history_us_steel.csv
   Структура: 25 строк, 16 колонок

✅ Загружено 25 статей из файла истории для 2024 года


### 📄 Таблица данных из файла истории (2024 год)

,№,Статья (файл истории),Значение (млн USD),Значение (USD)
0,1,amortization,$20.00,"$20,000,000"
1,2,cogs,"$14,060.00","$14,060,000,000"
2,3,depreciation_finance_leases,$56.00,"$56,000,000"
3,4,depreciation_owned,"$-14,084.00","$-14,084,000,000"
4,5,depreciation_rou,$14.00,"$14,000,000"
5,6,ebit,$240.00,"$240,000,000"
6,7,ebitda,$422.00,"$422,000,000"
7,8,ebt,$440.00,"$440,000,000"
8,9,eps_basic,$0.00,$2
9,10,eps_diluted,$0.00,$2


## 🗄️ ШАГ 3: Загрузка данных из БД (canonical формат)

In [26]:
# === Загрузка данных из БД: RAW (до маппинга) и CANONICAL (после маппинга) ===

mart = get_data_mart(ROOT, COMPANY)

# RAW данные - как они загрузились в БД из файла истории (БЕЗ маппинга)
raw_is = mart.get_history_income_statement(canonical=False)
print("📥 RAW данные из БД (до маппинга):")
print(f"   Загружено: {raw_is.shape[0]} строк, {raw_is.shape[1]} колонок")

# CANONICAL данные - после применения маппинга в DataMart
canonical_is = mart.get_history_income_statement(canonical=True)
print(f"\n📤 CANONICAL данные из БД (после маппинга):")
print(f"   Загружено: {canonical_is.shape[0]} строк, {canonical_is.shape[1]} колонок")

mart.close()

# Получаем данные 2024 из RAW формата (до маппинга)
raw_2024 = {}
if not raw_is.empty:
    if 'year' in raw_is.columns and 'metric' in raw_is.columns:
        # Long format
        is_2024_raw = raw_is[raw_is['year'] == 2024]
        for _, row in is_2024_raw.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['value'], errors='coerce')
            if pd.notna(value):
                raw_2024[metric] = float(value)
    elif 'metric' in raw_is.columns:
        # Wide format
        if '2024' in raw_is.columns:
            for _, row in raw_is.iterrows():
                metric = row['metric']
                value = pd.to_numeric(row['2024'], errors='coerce')
                if pd.notna(value):
                    raw_2024[metric] = float(value)
    
    print(f"\n✅ Загружено {len(raw_2024)} статей из RAW формата для 2024 года")
    
    # Создаём таблицу для отображения RAW данных
    raw_df_display = pd.DataFrame([
        {
            '№': i + 1,
            'Статья (RAW - из файла истории)': item,
            'Значение (млн USD)': value / 1_000_000,
            'Значение (USD)': value
        }
        for i, (item, value) in enumerate(sorted(raw_2024.items()))
    ])
    
    display(Markdown("### 📥 Таблица RAW данных из БД (2024 год, до маппинга)"))
    display(raw_df_display.style.format({
        'Значение (млн USD)': '${:,.2f}',
        'Значение (USD)': '${:,.0f}'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '9px'}))
else:
    print("⚠️ RAW IS пуст или не найден в БД")

# Получаем данные 2024 из canonical формата (после маппинга)
canonical_2024 = {}
if not canonical_is.empty:
    print(f"\n✅ Canonical IS загружен из БД: {canonical_is.shape[0]} записей")
    
    if 'year' in canonical_is.columns and 'metric' in canonical_is.columns:
        # Long format
        is_2024 = canonical_is[canonical_is['year'] == 2024]
        for _, row in is_2024.iterrows():
            metric = row['metric']
            value = pd.to_numeric(row['value'], errors='coerce')
            if pd.notna(value):
                canonical_2024[metric] = float(value)
    elif 'metric' in canonical_is.columns:
        # Wide format
        if '2024' in canonical_is.columns:
            for _, row in canonical_is.iterrows():
                metric = row['metric']
                value = pd.to_numeric(row['2024'], errors='coerce')
                if pd.notna(value):
                    canonical_2024[metric] = float(value)
    
    print(f"\n✅ Загружено {len(canonical_2024)} статей из canonical формата для 2024 года")
    
    # Создаём таблицу для отображения canonical данных
    canonical_df_display = pd.DataFrame([
        {
            '№': i + 1,
            'Статья (canonical - после маппинга)': item,
            'Значение (млн USD)': value / 1_000_000,
            'Значение (USD)': value
        }
        for i, (item, value) in enumerate(sorted(canonical_2024.items()))
    ])
    
    display(Markdown("### 📤 Таблица CANONICAL данных из БД (2024 год, после маппинга)"))
    display(canonical_df_display.style.format({
        'Значение (млн USD)': '${:,.2f}',
        'Значение (USD)': '${:,.0f}'
    }).set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '9px'}))
    
    # Сравнение RAW vs CANONICAL
    print("\n" + "="*80)
    print("🔍 Сравнение RAW vs CANONICAL:")
    print("="*80)
    
    comparison_raw_canonical = []
    all_metrics = set(raw_2024.keys()) | set(canonical_2024.keys())
    
    for metric in sorted(all_metrics):
        raw_val = raw_2024.get(metric)
        canonical_val = canonical_2024.get(metric)
        
        if raw_val is not None and canonical_val is not None:
            diff = abs(raw_val - canonical_val)
            if diff > 1000:  # Разница больше $1K
                comparison_raw_canonical.append({
                    'Метрика': metric,
                    'RAW значение': f"${raw_val:,.0f}",
                    'CANONICAL значение': f"${canonical_val:,.0f}",
                    'Разница': f"${diff:,.0f}",
                    'Статус': '⚠️ Изменено'
                })
            else:
                comparison_raw_canonical.append({
                    'Метрика': metric,
                    'RAW значение': f"${raw_val:,.0f}",
                    'CANONICAL значение': f"${canonical_val:,.0f}",
                    'Разница': f"${diff:,.0f}",
                    'Статус': '✅ Совпадает'
                })
        elif raw_val is not None:
            comparison_raw_canonical.append({
                'Метрика': metric,
                'RAW значение': f"${raw_val:,.0f}",
                'CANONICAL значение': '-',
                'Разница': '-',
                'Статус': '⚠️ Только в RAW'
            })
        elif canonical_val is not None:
            comparison_raw_canonical.append({
                'Метрика': metric,
                'RAW значение': '-',
                'CANONICAL значение': f"${canonical_val:,.0f}",
                'Разница': '-',
                'Статус': 'ℹ️ Добавлено в canonical'
            })
    
    if comparison_raw_canonical:
        comparison_df_raw_canon = pd.DataFrame(comparison_raw_canonical)
        display(Markdown("#### 📊 Сравнение RAW vs CANONICAL:"))
        display(comparison_df_raw_canon.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '9px'}))
        
        changed = len([x for x in comparison_raw_canonical if 'Изменено' in x['Статус']])
        added = len([x for x in comparison_raw_canonical if 'Добавлено' in x['Статус']])
        print(f"\n📋 Сводка:")
        print(f"  Метрик изменено при маппинге: {changed}")
        print(f"  Метрик добавлено в canonical: {added}")
        print(f"  Метрик совпадает: {len(comparison_raw_canonical) - changed - added}")
else:
    print("⚠️ Canonical IS пуст или не найден в БД")

📥 RAW данные из БД (до маппинга):
   Загружено: 25 строк, 16 колонок

📤 CANONICAL данные из БД (после маппинга):
   Загружено: 22 строк, 16 колонок

✅ Загружено 25 статей из RAW формата для 2024 года


### 📥 Таблица RAW данных из БД (2024 год, до маппинга)

,№,Статья (RAW - из файла истории),Значение (млн USD),Значение (USD)
0,1,amortization,$20.00,"$20,000,000"
1,2,cogs,"$14,060.00","$14,060,000,000"
2,3,depreciation_finance_leases,$56.00,"$56,000,000"
3,4,depreciation_owned,"$-14,084.00","$-14,084,000,000"
4,5,depreciation_rou,$14.00,"$14,000,000"
5,6,ebit,$240.00,"$240,000,000"
6,7,ebitda,$422.00,"$422,000,000"
7,8,ebt,$438.00,"$438,000,000"
8,9,eps_basic,$0.00,$2
9,10,eps_diluted,$0.00,$2



✅ Canonical IS загружен из БД: 22 записей

✅ Загружено 22 статей из canonical формата для 2024 года


### 📤 Таблица CANONICAL данных из БД (2024 год, после маппинга)

,№,Статья (canonical - после маппинга),Значение (млн USD),Значение (USD)
0,1,amortization,$20.00,"$20,000,000"
1,2,cogs,"$14,060.00","$14,060,000,000"
2,3,depreciation_owned,"$-14,084.00","$-14,084,000,000"
3,4,depreciation_rou,$14.00,"$14,000,000"
4,5,ebit,$240.00,"$240,000,000"
5,6,ebitda,$422.00,"$422,000,000"
6,7,ebt,$438.00,"$438,000,000"
7,8,eps_basic,$0.00,$2
8,9,eps_diluted,$0.00,$2
9,10,gain_loss_on_disposal,$0.00,$0



🔍 Сравнение RAW vs CANONICAL:


#### 📊 Сравнение RAW vs CANONICAL:

,Метрика,RAW значение,CANONICAL значение,Разница,Статус
0,amortization,"$20,000,000","$20,000,000",$0,✅ Совпадает
1,cogs,"$14,060,000,000","$14,060,000,000",$0,✅ Совпадает
2,depreciation_finance_leases,"$56,000,000",-,-,⚠️ Только в RAW
3,depreciation_owned,"$-14,084,000,000","$-14,084,000,000",$0,✅ Совпадает
4,depreciation_rou,"$14,000,000","$14,000,000",$0,✅ Совпадает
5,ebit,"$240,000,000","$240,000,000",$0,✅ Совпадает
6,ebitda,"$422,000,000","$422,000,000",$0,✅ Совпадает
7,ebt,"$438,000,000","$438,000,000",$0,✅ Совпадает
8,eps_basic,$2,$2,$0,✅ Совпадает
9,eps_diluted,$2,$2,$0,✅ Совпадает



📋 Сводка:
  Метрик изменено при маппинге: 0
  Метрик добавлено в canonical: 0
  Метрик совпадает: 25


## 🔗 ШАГ 4: Загрузка маппинга статей

**Примечание**: Маппинг статей из официального отчета на canonical метрики происходит автоматически в препроцессинге при загрузке данных в БД через DataMart. 

Здесь мы загружаем конфигурацию маппинга из `excel_loader.yaml` только для **понимания**, какие статьи из официального отчета соответствуют каким canonical метрикам. Это нужно для:
- Сопоставления данных из официального отчета с canonical метриками
- Понимания, какие статьи были объединены или переименованы
- Проверки правильности маппинга

**Важно**: Если данные уже загружены в БД, маппинг уже применён. Наша задача - убедиться, что значения в файле истории соответствуют официальной отчетности.

In [27]:
# === Загрузка конфигурации маппинга из excel_loader.yaml ===
# Примечание: Маппинг для IS находится в excel_loader.yaml (не в отдельном is_mapping.yaml)
# Это файл для загрузки данных из Excel, который содержит маппинг метрик

excel_loader_path = croot / "configs" / "excel_loader.yaml"
mapping_config = {}
required_metrics = {}

if excel_loader_path.exists():
    print(f"✅ Файл найден: {excel_loader_path.relative_to(ROOT)}")
    with open(excel_loader_path, 'r', encoding='utf-8') as f:
        mapping_config = yaml.safe_load(f)
    
    # Проверяем структуру файла
    print(f"\n📋 Доступные секции в excel_loader.yaml: {list(mapping_config.keys())}")
    
    # Ищем секцию IS
    is_mapping = mapping_config.get('history', {}).get('IS', {})
    if not is_mapping:
        # Пробуем напрямую
        is_mapping = mapping_config.get('IS', {})
    
    if is_mapping:
        print("✅ Найдена конфигурация маппинга для IS")
        required_metrics = is_mapping.get('required_metrics', {})
        
        if required_metrics:
            print(f"\n📊 Найдено {len(required_metrics)} метрик в маппинге")
            print(f"\nТребуемые метрики (с алиасами):")
            
            metrics_with_aliases = []
            metrics_without_aliases = []
            
            for metric, config in required_metrics.items():
                if isinstance(config, dict):
                    aliases = config.get('aliases', [])
                    if aliases:
                        metrics_with_aliases.append((metric, aliases))
                    else:
                        metrics_without_aliases.append(metric)
                else:
                    metrics_without_aliases.append(metric)
            
            # Показываем метрики с алиасами
            if metrics_with_aliases:
                print("\n  Метрики с алиасами:")
                for metric, aliases in metrics_with_aliases[:15]:
                    print(f"    {metric}: {', '.join(aliases[:3])}{'...' if len(aliases) > 3 else ''}")
            
            # Показываем метрики без алиасов
            if metrics_without_aliases:
                print(f"\n  Метрики без алиасов ({len(metrics_without_aliases)}): {', '.join(metrics_without_aliases[:10])}{'...' if len(metrics_without_aliases) > 10 else ''}")
        else:
            print("⚠️ Секция 'required_metrics' не найдена в конфигурации IS")
    else:
        print("⚠️ Конфигурация IS не найдена в excel_loader.yaml")
        print(f"   Доступные секции: {list(mapping_config.keys())}")
        if 'history' in mapping_config:
            print(f"   Секции в 'history': {list(mapping_config['history'].keys())}")
else:
    print(f"⚠️ Файл excel_loader.yaml не найден: {excel_loader_path}")
    print(f"   Проверьте путь: {croot / 'configs'}")

print(f"\n✅ Маппинг загружен: {len(required_metrics)} метрик")

✅ Файл найден: companies/us_steel/configs/excel_loader.yaml

📋 Доступные секции в excel_loader.yaml: ['meta', 'history', 'schedules', 'segments', 'operational_drivers', 'macro', 'validation', 'notes', 'loader_defaults']
✅ Найдена конфигурация маппинга для IS

📊 Найдено 31 метрик в маппинге

Требуемые метрики (с алиасами):

  Метрики с алиасами:
    revenue: sales_revenue_net, revenues, net_sales...
    gross_profit: gross_profit
    cogs: cost_of_goods_sold, cost_of_goods_and_services_sold, cost_of_revenue...
    sga: selling_general_and_administrative_expense, selling_and_marketing_expense, general_and_administrative_expense
    rnd: research_and_development_expense
    depreciation_owned: depreciation, depreciationexpense, depreciation_depletion_and_amortization...
    depreciation_rou: operating_lease_cost, operating_lease_right_of_use_asset_depreciation_expense
    amortization: amortization_of_intangible_assets, amortization_intangibles
    total_da: total_depreciation_amortizatio

## 🔍 ШАГ 5: Детальное сравнение всех источников

**Цель**: Сравнить данные из официального отчета с данными в БД на разных этапах обработки:
1. **Официальный отчет** → исходные данные из отчетности
2. **RAW данные из БД** → как данные загрузились в БД из CSV (до маппинга)
3. **CANONICAL данные из БД** → как данные стали после маппинга в DataMart (препроцессинг)

**Опционально**: Файл истории (CSV) - для проверки, что CSV правильно загрузился в БД

Это позволит понять:
- Правильно ли данные из официального отчета попали в БД (RAW)
- Правильно ли препроцессинг применил маппинг (RAW → CANONICAL)
- Где происходят расхождения и что нужно исправить

In [28]:
# === Детальное сравнение всех источников с учетом маппинга ===

def similarity(a, b):
    """Вычисляет схожесть двух строк (0-1)"""
    return SequenceMatcher(None, a.lower(), b.lower()).ratio()

def normalize_name(name):
    """Нормализует название для сравнения, возвращает кортеж (с пробелами, без пробелов)"""
    if not name:
        return ("", "")
    # Удаляем специальные символы, приводим к нижнему регистру
    normalized = re.sub(r'[^\w\s]', '', str(name).lower())
    # Удаляем лишние пробелы
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    # Также создаем версию без пробелов для сопоставления с CamelCase из EDGAR
    normalized_no_spaces = normalized.replace(' ', '')
    return (normalized, normalized_no_spaces)

def normalize_name_simple(name):
    """Простая нормализация для обратной совместимости, возвращает строку"""
    if not name:
        return ""
    normalized = re.sub(r'[^\w\s]', '', str(name).lower())
    normalized = re.sub(r'\s+', ' ', normalized).strip()
    return normalized

def find_best_match(official_name, candidate_list, threshold=0.7):
    """Находит лучшее совпадение для названия из официального отчета"""
    best_match = None
    best_score = 0.0
    
    normalized_official_tuple = normalize_name(official_name)
    normalized_official = normalized_official_tuple[0] if isinstance(normalized_official_tuple, tuple) else normalized_official_tuple
    normalized_official_no_spaces = normalized_official_tuple[1] if isinstance(normalized_official_tuple, tuple) else normalized_official.replace(' ', '')
    
    for candidate in candidate_list:
        normalized_candidate_tuple = normalize_name(candidate)
        normalized_candidate = normalized_candidate_tuple[0] if isinstance(normalized_candidate_tuple, tuple) else normalized_candidate_tuple
        normalized_candidate_no_spaces = normalized_candidate_tuple[1] if isinstance(normalized_candidate_tuple, tuple) else normalized_candidate.replace(' ', '')
        
        # Проверяем совпадение с пробелами и без
        score1 = similarity(normalized_official, normalized_candidate)
        score2 = similarity(normalized_official_no_spaces, normalized_candidate_no_spaces)
        score = max(score1, score2)
        
        if normalized_official == normalized_candidate or normalized_official_no_spaces == normalized_candidate_no_spaces:
            return candidate, 1.0
        
        if score > best_score:
            best_score = score
            best_match = candidate
    
    if best_score >= threshold:
        return best_match, best_score
    return None, best_score

display(Markdown("### 🔍 Детальное сравнение данных 2024"))

if not official_reporting_2024:
    print("❌ Данные из официальной отчетности не загружены. Запустите ячейку ШАГ 1.")
elif 'raw_2024' not in globals() or not raw_2024:
    print("❌ RAW данные из БД не загружены. Запустите ячейку ШАГ 3.")
elif not canonical_2024:
    print("❌ Данные из canonical формата не загружены. Запустите ячейку ШАГ 3.")
else:
    print(f"✅ Основные источники данных загружены:")
    print(f"  1. Официальный отчет: {len(official_reporting_2024)} статей")
    print(f"  2. RAW из БД (до маппинга): {len(raw_2024)} статей")
    print(f"  3. CANONICAL из БД (после маппинга): {len(canonical_2024)} статей")
    
    # Опционально: файл истории
    if 'hist_2024_raw' in globals() and hist_2024_raw:
        print(f"  4. Файл истории (опционально): {len(hist_2024_raw)} статей")
    else:
        print(f"  4. Файл истории: не загружен (опционально)")
    
    # Создаём маппинг алиасов (alias -> canonical)
    aliases_map = {}
    # Обратный маппинг (canonical -> aliases) для поиска
    canonical_to_aliases = {}
    
    for metric, config in required_metrics.items():
        aliases = config.get('aliases', [])
        canonical_to_aliases[metric] = aliases
        for alias in aliases:
            aliases_map[alias.lower()] = metric
        aliases_map[metric.lower()] = metric
    
    # Создаём маппинг официальных названий на canonical метрики для Income Statement
    # На основе типичных названий из отчетности, маппинга из YAML и алиасов из EDGAR loader
    # Источник алиасов: /Users/arturhusnutdinov/Documents/IT Development/Docker/EDGAR/edgar_loader/loader.py
    
    official_to_canonical_map = {
        # === REVENUE ===
        # Из EDGAR: "revenue": ["NetSales", "SalesRevenueNet", "Revenues", "RevenueFromContractWithCustomerExcludingAssessedTax"]
        'Net sales': 'revenue',
        'Net sales to related parties': 'revenue',  # Объединяется с Net sales → revenue
        'Total net sales': 'revenue',
        'Sales revenue net': 'revenue',
        'Revenue from contract with customer excluding assessed tax': 'revenue',
        'Revenues': 'revenue',
        
        # === RELATED PARTY REVENUE ===
        # Из EDGAR: "related_party_revenue": ["RevenueFromRelatedParties", "NetSalesToRelatedParties"]
        'Net sales to related parties': 'revenue',  # Объединяется в revenue
        
        # === COSTS & EXPENSES ===
        # Из EDGAR: "cogs": ["CostOfGoodsAndServicesSold", "CostOfRevenue", "CostOfSales", "CostOfGoodsSold"]
        'Cost of sales (excludes items shown below)': 'cogs',
        'Cost of goods sold': 'cogs',
        'Cost of goods and services sold': 'cogs',
        'Cost of revenue': 'cogs',
        'Cost of sales': 'cogs',
        
        # Из EDGAR: "sga": ["SellingGeneralAndAdministrativeExpense", "SellingAndMarketingExpense", "GeneralAndAdministrativeExpense"]
        'Selling, general and administrative expenses': 'sga',
        'Selling general and administrative expense': 'sga',
        'Selling and marketing expense': 'sga',
        'General and administrative expense': 'sga',
        
        # === DEPRECIATION & AMORTIZATION ===
        # Из EDGAR: "depreciation_amortization": ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization", "Depreciation", "DepreciationExpense"]
        'Depreciation, depletion and amortization': 'total_da',  # ВАЖНО: это total_da, не depreciation_owned!
        'Depreciation and amortization': 'total_da',
        'Depreciation depletion and amortization': 'total_da',
        'Depreciation (owned assets)': 'depreciation_owned',
        'Depreciation expense': 'depreciation_owned',
        'Depreciation': 'depreciation_owned',
        
        # Из EDGAR: "depreciation_rou": ["OperatingLeaseCost", "OperatingLeaseRightOfUseAssetDepreciationExpense"]
        'Depreciation (ROU assets)': 'depreciation_rou',
        'Operating lease cost': 'depreciation_rou',
        'Operating lease right of use asset depreciation expense': 'depreciation_rou',
        
        # Из EDGAR: "amortization_intangibles": ["AmortizationOfIntangibleAssets"]
        'Amortization': 'amortization',
        'Amortization of intangible assets': 'amortization',
        
        # === OPERATING INCOME ===
        # Из EDGAR: "ebit": ["OperatingIncomeLoss", "EarningsBeforeInterestAndTaxes"]
        'Earnings before interest and income taxes (EBIT)': 'ebit',
        'Earnings before interest and income taxes': 'ebit',
        'Operating income (EBIT)': 'ebit',
        'Operating income loss': 'ebit',
        'EBIT': 'ebit',
        
        # Из EDGAR: "ebitda": ["EarningsBeforeInterestTaxesDepreciationAndAmortization"]
        'EBITDA': 'ebitda',
        'Earnings before interest taxes depreciation and amortization': 'ebitda',
        
        # === OTHER OPERATING ITEMS ===
        # Из EDGAR: "earnings_from_investees": ["IncomeLossFromEquityMethodInvestments", "EarningsFromInvestees"]
        'Earnings from investees': 'earnings_from_investees',
        'Income loss from equity method investments': 'earnings_from_investees',
        
        # Из EDGAR: "asset_impairment": ["AssetImpairmentCharges", "ImpairmentOfLongLivedAssetsToBeDisposed"]
        'Asset impairment charges': 'asset_impairment',
        'Impairment of long lived assets to be disposed': 'asset_impairment',
        
        # Из EDGAR: "restructuring_charges": ["RestructuringCharges", "RestructuringAndOtherCharges"]
        'Restructuring and other charges': 'restructuring_charges',
        'Restructuring charges': 'restructuring_charges',
        
        # Из EDGAR: "other_gains_losses": ["OtherGainsLossesNet", "NonoperatingIncomeExpense"]
        'Other losses (gains), net': 'other_gains_losses',
        'Other gains losses net': 'other_gains_losses',
        'Nonoperating income expense': 'other_gains_losses',
        
        # Из EDGAR: "gain_loss_disposal": ["GainLossOnSaleOfPropertyPlantEquipment"]
        'Gain (loss) on disposal of assets': 'gain_loss_on_disposal',
        'Gain loss on sale of property plant equipment': 'gain_loss_on_disposal',
        'Gain on equity investee transactions': 'gain_loss_on_disposal',  # Может быть частью gain_loss_on_disposal
        
        # === INTEREST & OTHER FINANCIAL ITEMS ===
        # Из EDGAR: "interest_expense": ["InterestExpense", "InterestExpenseDebt", "InterestAndDebtExpense"]
        'Interest expense': 'interest_expense',
        'Interest expense debt': 'interest_expense',
        'Interest and debt expense': 'interest_expense',
        
        # Из EDGAR: "interest_income": ["InvestmentIncomeInterest", "InterestIncome"]
        # ВАЖНО: в отчете может быть отрицательным (доход), но в canonical положительное
        'Interest income': 'interest_income',
        'Investment income interest': 'interest_income',
        
        # Из EDGAR: "debt_extinguishment_loss": ["GainsLossesOnExtinguishmentOfDebt", "LossOnDebtExtinguishment"]
        'Loss on debt extinguishment': 'debt_extinguishment_loss',
        'Gains losses on extinguishment of debt': 'debt_extinguishment_loss',
        
        # Из EDGAR: "other_financial_costs": ["OtherFinancialCosts"]
        'Other financial costs': 'other_financial_costs',
        
        # Из EDGAR: "benefit_income": ["NetPeriodicBenefitCost", "NetPeriodicBenefitIncome"]
        'Net periodic benefit income': 'benefit_income',
        'Net periodic benefit cost': 'benefit_income',
        
        # Из EDGAR: "gain_loss_investments": ["GainLossOnInvestmentsRelatedToActiveEmployeeBenefits"]
        'Net gain from investments related to active employee benefits': 'gain_loss_investments',
        'Gain loss on investments related to active employee benefits': 'gain_loss_investments',
        
        # Из EDGAR: "lease_interest": (может быть частью interest_expense или отдельно)
        'Lease interest expense': 'lease_interest',
        
        # Суммарная метрика
        'Net interest and other financial (benefits) costs': 'interest_income',  # Может быть частью interest_income
        
        # === INCOME BEFORE TAXES ===
        # Из EDGAR: "ebt": ["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest", "IncomeLossFromContinuingOperationsBeforeIncomeTaxes"]
        'Earnings before income taxes (EBT)': 'ebt',
        'Income before income taxes (EBT)': 'ebt',
        'Income loss from continuing operations before income taxes': 'ebt',
        
        # === TAXES ===
        # Из EDGAR: "tax_expense": ["IncomeTaxExpenseBenefit"]
        # ВАЖНО: в отчете положительное (расход), в canonical тоже положительное
        'Income tax expense': 'tax_expense',
        'Tax expense': 'tax_expense',
        'Income tax expense benefit': 'tax_expense',
        
        # === NET INCOME ===
        # Из EDGAR: "net_income": ["NetIncomeLoss", "ProfitLoss"]
        'Net earnings': 'net_income',
        'Net income': 'net_income',
        'Net income loss': 'net_income',
        'Profit loss': 'net_income',
        
        # Из EDGAR: "net_income_attributable_to_parent": ["NetIncomeLossAttributableToParent", "NetIncomeLossAttributableToUnitedStatesSteelCorporation"]
        'Net earnings attributable to United States Steel Corporation': 'net_income',
        'Net income loss attributable to parent': 'net_income',
        'Net income loss attributable to United States Steel Corporation': 'net_income',
        
        # Из EDGAR: "nci_income": ["NetIncomeLossAttributableToNoncontrollingInterest"]
        'Net earnings attributable to noncontrolling interests': 'nci_income',
        'Net income loss attributable to noncontrolling interest': 'nci_income',
        
        # === EPS ===
        # Из EDGAR: "eps_basic": ["EarningsPerShareBasic"]
        'Earnings per share (basic)': 'eps_basic',
        'Earnings per share basic': 'eps_basic',
        
        # Из EDGAR: "eps_diluted": ["EarningsPerShareDiluted"]
        'Earnings per share (diluted)': 'eps_diluted',
        'Earnings per share diluted': 'eps_diluted',
    }
    
    # Создаём сравнительную таблицу
    comparison_data = []
    
    for official_item, official_val in official_reporting_2024.items():
        # Ищем совпадение в файле истории (опционально)
        # Используем тот же маппинг, что и для RAW данных
        hist_match = None
        if 'hist_2024_raw' in globals() and hist_2024_raw:
            # Сначала проверяем явный маппинг
            if official_item in official_to_canonical_map:
                canonical_name = official_to_canonical_map[official_item]
                # Ищем canonical метрику в файле истории
                if canonical_name in hist_2024_raw:
                    hist_match = canonical_name
                # Также проверяем алиасы
                if not hist_match and canonical_name in canonical_to_aliases:
                    for alias in canonical_to_aliases[canonical_name]:
                        if alias in hist_2024_raw:
                            hist_match = alias
                            break
            
            # Если не нашли через маппинг, ищем по названию
            # НО исключаем старые/неправильные метрики
            if not hist_match:
                # Исключаем старые метрики, которые могут быть перепутаны
                excluded_hist_metrics = set()
                
                # Для noncurrent lease liabilities не ищем current_lease_liability
                if 'noncurrent' in official_item.lower() and 'lease' in official_item.lower():
                    excluded_hist_metrics.add('current_lease_liability')
                    excluded_hist_metrics.add('lease_liab_current')
                
                # Для accounts_payable не ищем accounts_payable_related_parties
                if 'accounts payable and other accrued liabilities' in official_item.lower():
                    excluded_hist_metrics.add('accounts_payable_related_parties')
                
                # Для dtl не ищем other_current_liabilities
                if 'deferred income tax liabilities' in official_item.lower():
                    excluded_hist_metrics.add('other_current_liabilities')
                
                available_hist_metrics = [m for m in hist_2024_raw.keys() if m not in excluded_hist_metrics]
                hist_match, hist_score = find_best_match(official_item, available_hist_metrics, threshold=0.6)
        
        # Ищем совпадение в RAW данных из БД
        # Сначала проверяем явный маппинг
        raw_match = None
        if official_item in official_to_canonical_map:
            canonical_name = official_to_canonical_map[official_item]
            # Ищем canonical метрику в RAW данных (она может быть там под своим именем)
            if canonical_name in raw_2024:
                raw_match = canonical_name
            # Также проверяем алиасы
            if not raw_match and canonical_name in canonical_to_aliases:
                for alias in canonical_to_aliases[canonical_name]:
                    if alias in raw_2024:
                        raw_match = alias
                        break
        
        # Если не нашли через маппинг, ищем по названию
        if not raw_match:
            raw_match, raw_score = find_best_match(official_item, raw_2024.keys(), threshold=0.6)
        
        # ПРИОРИТЕТ 1: Используем явный маппинг официальных названий
        explicit_canonical_match = official_to_canonical_map.get(official_item)
        
        # ПРИОРИТЕТ 2: Ищем через алиасы (нормализуем название)
        alias_match = None
        official_normalized_tuple = normalize_name(official_item)
        official_normalized = official_normalized_tuple[0] if isinstance(official_normalized_tuple, tuple) else official_normalized_tuple
        official_normalized_no_spaces = official_normalized_tuple[1] if isinstance(official_normalized_tuple, tuple) else official_normalized.replace(' ', '')
        
        # Проверяем прямое совпадение с алиасами (с пробелами и без)
        if official_normalized in aliases_map:
            alias_match = aliases_map[official_normalized]
        elif official_normalized_no_spaces in aliases_map:
            alias_match = aliases_map[official_normalized_no_spaces]
        else:
            # Проверяем частичное совпадение с алиасами
            for alias, canonical in aliases_map.items():
                alias_normalized = normalize_name_simple(alias).replace(' ', '')
                if (official_normalized in alias or alias in official_normalized or
                    official_normalized_no_spaces in alias_normalized or alias_normalized in official_normalized_no_spaces):
                    alias_match = canonical
                    break
        
        # ПРИОРИТЕТ 3: Проверяем алиасы canonical метрик
        alias_from_canonical = None
        if not explicit_canonical_match and not alias_match:
            for canonical_metric, aliases_list in canonical_to_aliases.items():
                for alias in aliases_list:
                    alias_normalized_tuple = normalize_name(alias)
                    alias_normalized = alias_normalized_tuple[0] if isinstance(alias_normalized_tuple, tuple) else alias_normalized_tuple
                    alias_normalized_no_spaces = alias_normalized_tuple[1] if isinstance(alias_normalized_tuple, tuple) else alias_normalized.replace(' ', '')
                    # Проверяем точное совпадение или что официальное название содержит алиас
                    if (official_normalized == alias_normalized or 
                        alias_normalized in official_normalized or
                        official_normalized_no_spaces == alias_normalized_no_spaces or
                        alias_normalized_no_spaces in official_normalized_no_spaces):
                        alias_from_canonical = canonical_metric
                        break
                if alias_from_canonical:
                    break
        
        # ПРИОРИТЕТ 4: Ищем совпадение по названию в canonical метриках
        # НО исключаем метрики, которые уже найдены через маппинг
        canonical_match = None
        canonical_score = 0.0
        if not explicit_canonical_match and not alias_match and not alias_from_canonical:
            # Исключаем метрики, которые могут быть перепутаны
            # Например, не ищем "cash" если ищем "restricted_cash"
            excluded_metrics = set()
            if 'restricted' in official_normalized or 'restricted' in official_item.lower():
                excluded_metrics.add('cash')  # Не путать restricted_cash с cash
            
            available_metrics = [m for m in canonical_2024.keys() if m not in excluded_metrics]
            canonical_match, canonical_score = find_best_match(official_item, available_metrics, threshold=0.6)
        
        # Используем найденное совпадение в порядке приоритета
        final_canonical_match = (explicit_canonical_match or 
                                alias_match or 
                                alias_from_canonical or 
                                canonical_match)
        
        # Проверяем, что найденная метрика существует в canonical данных
        if final_canonical_match and final_canonical_match not in canonical_2024:
            final_canonical_match = None
        
        # Получаем значения (файл истории опционален)
        hist_val = None
        if 'hist_2024_raw' in globals() and hist_2024_raw and hist_match:
            hist_val = hist_2024_raw.get(hist_match)
        
        raw_val = raw_2024.get(raw_match) if raw_match else None
        canonical_val = canonical_2024.get(final_canonical_match) if final_canonical_match else None
        
        # Проверяем расхождения с учетом знаков
        issues = []
        
        # Определяем метрики, которые могут иметь разные знаки в отчете и canonical
        # (например, interest_income в отчете отрицательное = доход, в canonical положительное)
        sign_sensitive_metrics = {
            'interest_income': True,  # В отчете отрицательное = доход, в canonical положительное
            'interest_expense': False,  # Всегда положительное
            'tax_expense': False,  # Всегда положительное (расход)
            'net_income': False,  # Может быть отрицательным в обоих
        }
        
        # Функция для нормализации знака при сравнении
        def normalize_sign_for_comparison(official_val, canonical_val, metric_name):
            """Нормализует знаки для сравнения"""
            if metric_name in sign_sensitive_metrics and sign_sensitive_metrics[metric_name]:
                # Для interest_income: отрицательное в отчете = положительное в canonical
                if official_val < 0:
                    official_val_normalized = abs(official_val)
                else:
                    official_val_normalized = official_val
                return official_val_normalized, abs(canonical_val) if canonical_val else None
            return official_val, canonical_val
        
        # Сравнение с файлом истории (опционально)
        if hist_match and hist_val is not None:
            official_norm, hist_norm = normalize_sign_for_comparison(official_val, hist_val, final_canonical_match or '')
            diff = abs(official_norm - hist_norm) if hist_norm is not None else abs(official_val - hist_val)
            diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
            if diff > 1000:
                sign_note = " (учтен знак)" if final_canonical_match in sign_sensitive_metrics else ""
                issues.append(f"Офиц. vs История: ${diff:,.0f} ({diff_pct:.1f}%){sign_note}")
        
        # Основное сравнение: Официальный отчет vs RAW БД
        if raw_match and raw_val is not None:
            official_norm, raw_norm = normalize_sign_for_comparison(official_val, raw_val, final_canonical_match or '')
            diff = abs(official_norm - raw_norm) if raw_norm is not None else abs(official_val - raw_val)
            diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
            if diff > 1000:
                sign_note = " (учтен знак)" if final_canonical_match in sign_sensitive_metrics else ""
                issues.append(f"Офиц. vs RAW БД: ${diff:,.0f} ({diff_pct:.1f}%){sign_note}")
        
        # Основное сравнение: Официальный отчет vs CANONICAL БД
        if final_canonical_match and canonical_val is not None:
            official_norm, canonical_norm = normalize_sign_for_comparison(official_val, canonical_val, final_canonical_match)
            diff = abs(official_norm - canonical_norm) if canonical_norm is not None else abs(official_val - canonical_val)
            diff_pct = (diff / abs(official_val) * 100) if official_val != 0 else 0
            if diff > 1000:
                sign_note = " (учтен знак)" if final_canonical_match in sign_sensitive_metrics else ""
                # Дополнительная проверка для проблемных метрик
                if final_canonical_match == 'depreciation_owned' and abs(canonical_val) > 1e10:
                    issues.append(f"Офиц. vs Canonical: ${diff:,.0f} ({diff_pct:.1f}%) ⚠️ КРИТИЧЕСКАЯ ОШИБКА: неправильное значение в БД!")
                else:
                    issues.append(f"Офиц. vs Canonical: ${diff:,.0f} ({diff_pct:.1f}%){sign_note}")
        
        # Проверяем расхождения между этапами обработки в БД
        if hist_val is not None and raw_val is not None:
            diff = abs(hist_val - raw_val)
            if diff > 1000:
                issues.append(f"История vs RAW БД: ${diff:,.0f}")
        
        # Ключевое сравнение: RAW vs CANONICAL (как препроцессинг изменил данные)
        if raw_val is not None and canonical_val is not None:
            diff = abs(raw_val - canonical_val)
            if diff > 1000:
                issues.append(f"RAW vs Canonical: ${diff:,.0f}")
        
        comparison_data.append({
            'Официальный отчет': official_item,
            'Значение (офиц.)': f"${official_val:,.0f}",
            'Совпадение (история)': hist_match if hist_match else "-",
            'Значение (история)': f"${hist_val:,.0f}" if hist_val is not None else "-",
            'Совпадение (RAW БД)': raw_match if raw_match else "-",
            'Значение (RAW БД)': f"${raw_val:,.0f}" if raw_val is not None else "-",
            'Совпадение (canonical)': final_canonical_match if final_canonical_match else "-",
            'Значение (canonical)': f"${canonical_val:,.0f}" if canonical_val is not None else "-",
            'Проблемы': '; '.join(issues) if issues else '✅'
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    # Показываем проблемные статьи
    problematic = comparison_df[comparison_df['Проблемы'] != '✅']
    if not problematic.empty:
        display(Markdown("#### ❌ Статьи с расхождениями:"))
        problematic_display = problematic[['Официальный отчет', 'Значение (офиц.)', 'Совпадение (история)', 
                             'Значение (история)', 'Совпадение (RAW БД)', 'Значение (RAW БД)',
                             'Совпадение (canonical)', 'Значение (canonical)', 'Проблемы']]
        display(problematic_display.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'table', 'props': [('font-size', '9px')]}
        ]).set_properties(**{'font-size': '9px'}))
    
    # Статьи, которые не были найдены в БД (основной критерий)
    not_found = comparison_df[(comparison_df['Совпадение (RAW БД)'] == '-') &
                               (comparison_df['Совпадение (canonical)'] == '-')]
    if not not_found.empty:
        display(Markdown("#### ⚠️ Статьи из официального отчета, не найденные в модели:"))
        not_found_display = not_found[['Официальный отчет', 'Значение (офиц.)']]
        display(not_found_display.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '4px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '9px'}))
    
    # Показываем полную таблицу с уменьшенным шрифтом
    display(Markdown("#### 📊 Полная таблица сравнения:"))
    display(comparison_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left'), ('white-space', 'nowrap')]},
        {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left'), ('white-space', 'nowrap')]},
        {'selector': 'table', 'props': [('font-size', '8px'), ('width', '100%')]}
    ]).set_properties(**{'font-size': '8px', 'max-width': '120px', 'overflow': 'hidden', 'text-overflow': 'ellipsis'}))
    
    # Сводка
    print(f"\n📋 Сводка сравнения:")
    print(f"  Всего статей в официальном отчете: {len(official_reporting_2024)}")
    
    # Подсчитываем совпадения
    hist_matches = len([x for x in comparison_df['Совпадение (история)'] if x != '-']) if 'hist_2024_raw' in globals() else 0
    raw_matches = len([x for x in comparison_df['Совпадение (RAW БД)'] if x != '-'])
    canonical_matches = len([x for x in comparison_df['Совпадение (canonical)'] if x != '-'])
    
    print(f"  Найдено совпадений в RAW БД: {raw_matches} (основной источник)")
    print(f"  Найдено совпадений в CANONICAL БД: {canonical_matches} (после маппинга)")
    if hist_matches > 0:
        print(f"  Найдено совпадений в файле истории: {hist_matches} (опционально)")
    
    print(f"  Статей с расхождениями: {len(problematic)}")
    print(f"  Статей не найденных в БД: {len(not_found)}")
    print(f"  Статей без проблем: {len(comparison_df) - len(problematic)}")
    
    # Дополнительная сводка по этапам обработки
    print(f"\n📊 Анализ этапов обработки:")
    
    # Подсчитываем изменения при загрузке в БД (история → RAW)
    hist_raw_diff = 0
    for _, row in comparison_df.iterrows():
        hist_match = row['Совпадение (история)']
        raw_match = row['Совпадение (RAW БД)']
        hist_val = row['Значение (история)']
        raw_val = row['Значение (RAW БД)']
        
        if hist_match != '-' and raw_match != '-' and hist_val != '-' and raw_val != '-':
            # Извлекаем числовые значения из строк вида "$1,234,567"
            try:
                hist_num = float(hist_val.replace('$', '').replace(',', ''))
                raw_num = float(raw_val.replace('$', '').replace(',', ''))
                if abs(hist_num - raw_num) > 1000:  # Разница больше $1K
                    hist_raw_diff += 1
            except (ValueError, AttributeError):
                pass
    
    # Подсчитываем изменения при маппинге (RAW → CANONICAL)
    raw_canonical_diff = 0
    for _, row in comparison_df.iterrows():
        raw_match = row['Совпадение (RAW БД)']
        canonical_match = row['Совпадение (canonical)']
        raw_val = row['Значение (RAW БД)']
        canonical_val = row['Значение (canonical)']
        
        if raw_match != '-' and canonical_match != '-' and raw_val != '-' and canonical_val != '-':
            try:
                raw_num = float(raw_val.replace('$', '').replace(',', ''))
                canonical_num = float(canonical_val.replace('$', '').replace(',', ''))
                if abs(raw_num - canonical_num) > 1000:  # Разница больше $1K
                    raw_canonical_diff += 1
            except (ValueError, AttributeError):
                pass
    
    print(f"  Изменений при загрузке в БД (история → RAW): {hist_raw_diff}")
    print(f"  Изменений при маппинге (RAW → CANONICAL): {raw_canonical_diff}")
    
    # Показываем примеры изменений
    if hist_raw_diff > 0:
        print(f"\n  Примеры изменений при загрузке в БД:")
        count = 0
        for _, row in comparison_df.iterrows():
            if count >= 3:
                break
            hist_match = row['Совпадение (история)']
            raw_match = row['Совпадение (RAW БД)']
            hist_val = row['Значение (история)']
            raw_val = row['Значение (RAW БД)']
            
            if hist_match != '-' and raw_match != '-' and hist_val != '-' and raw_val != '-':
                try:
                    hist_num = float(hist_val.replace('$', '').replace(',', ''))
                    raw_num = float(raw_val.replace('$', '').replace(',', ''))
                    if abs(hist_num - raw_num) > 1000:
                        print(f"    {row['Официальный отчет']}: История=${hist_num:,.0f} → RAW=${raw_num:,.0f} (diff=${abs(hist_num-raw_num):,.0f})")
                        count += 1
                except (ValueError, AttributeError):
                    pass
    
    if raw_canonical_diff > 0:
        print(f"\n  Примеры изменений при маппинге:")
        count = 0
        for _, row in comparison_df.iterrows():
            if count >= 3:
                break
            raw_match = row['Совпадение (RAW БД)']
            canonical_match = row['Совпадение (canonical)']
            raw_val = row['Значение (RAW БД)']
            canonical_val = row['Значение (canonical)']
            
            if raw_match != '-' and canonical_match != '-' and raw_val != '-' and canonical_val != '-':
                try:
                    raw_num = float(raw_val.replace('$', '').replace(',', ''))
                    canonical_num = float(canonical_val.replace('$', '').replace(',', ''))
                    if abs(raw_num - canonical_num) > 1000:
                        print(f"    {row['Официальный отчет']}: RAW=${raw_num:,.0f} → CANONICAL=${canonical_num:,.0f} (diff=${abs(raw_num-canonical_num):,.0f})")
                        count += 1
                except (ValueError, AttributeError):
                    pass

### 🔍 Детальное сравнение данных 2024

✅ Основные источники данных загружены:
  1. Официальный отчет: 27 статей
  2. RAW из БД (до маппинга): 25 статей
  3. CANONICAL из БД (после маппинга): 22 статей
  4. Файл истории (опционально): 25 статей


#### ❌ Статьи с расхождениями:

,Официальный отчет,Значение (офиц.),Совпадение (история),Значение (история),Совпадение (RAW БД),Значение (RAW БД),Совпадение (canonical),Значение (canonical),Проблемы
0,Net sales,"$13,135,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000","Офиц. vs История: $2,505,000,000 (19.1%); Офиц. vs RAW БД: $2,505,000,000 (19.1%); Офиц. vs Canonical: $2,505,000,000 (19.1%)"
1,Net sales to related parties,"$2,505,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000","Офиц. vs История: $13,135,000,000 (524.4%); Офиц. vs RAW БД: $13,135,000,000 (524.4%); Офиц. vs Canonical: $13,135,000,000 (524.4%)"
4,"Selling, general and administrative expenses","$435,000,000",sga,"$512,000,000",sga,"$435,000,000",sga,"$435,000,000","Офиц. vs История: $77,000,000 (17.7%); История vs RAW БД: $77,000,000"
11,Total operating expenses,"$15,400,000,000",other_expenses,"$15,456,000,000",other_expenses,"$15,456,000,000",other_expenses,"$15,456,000,000","Офиц. vs История: $56,000,000 (0.4%); Офиц. vs RAW БД: $56,000,000 (0.4%); Офиц. vs Canonical: $56,000,000 (0.4%)"
14,Interest income,"$-96,000,000",interest_income,"$-426,000,000",interest_income,"$-96,000,000",interest_income,"$-96,000,000","Офиц. vs История: $330,000,000 (343.8%) (учтен знак); История vs RAW БД: $330,000,000"
17,Net periodic benefit income,"$-132,000,000",interest_income,"$-426,000,000",interest_income,"$-96,000,000",-,-,"Офиц. vs История: $294,000,000 (222.7%); Офиц. vs RAW БД: $36,000,000 (27.3%); История vs RAW БД: $330,000,000"
19,Net interest and other financial (benefits) costs,"$-198,000,000",interest_income,"$-426,000,000",interest_income,"$-96,000,000",interest_income,"$-96,000,000","Офиц. vs История: $228,000,000 (115.2%) (учтен знак); Офиц. vs RAW БД: $102,000,000 (51.5%) (учтен знак); Офиц. vs Canonical: $102,000,000 (51.5%) (учтен знак); История vs RAW БД: $330,000,000"
20,Earnings before income taxes (EBT),"$438,000,000",ebt,"$440,000,000",ebt,"$438,000,000",ebt,"$438,000,000","Офиц. vs История: $2,000,000 (0.5%); История vs RAW БД: $2,000,000"
22,Net earnings,"$384,000,000",net_income,"$768,000,000",net_income,"$384,000,000",net_income,"$384,000,000","Офиц. vs История: $384,000,000 (100.0%) (учтен знак); История vs RAW БД: $384,000,000"
24,Net earnings attributable to United States Steel Corporation,"$384,000,000",net_income,"$768,000,000",net_income,"$384,000,000",net_income,"$384,000,000","Офиц. vs История: $384,000,000 (100.0%) (учтен знак); История vs RAW БД: $384,000,000"


#### ⚠️ Статьи из официального отчета, не найденные в модели:

,Официальный отчет,Значение (офиц.)
6,Earnings from investees,"$-112,000,000"
7,Asset impairment charges,"$19,000,000"
8,Restructuring and other charges,"$8,000,000"
10,"Other losses (gains), net","$77,000,000"
15,Loss on debt extinguishment,"$2,000,000"
16,Other financial costs,"$29,000,000"
18,Net gain from investments related to active employee benefits,"$-25,000,000"
23,Net earnings attributable to noncontrolling interests,$0


#### 📊 Полная таблица сравнения:

,Официальный отчет,Значение (офиц.),Совпадение (история),Значение (история),Совпадение (RAW БД),Значение (RAW БД),Совпадение (canonical),Значение (canonical),Проблемы
0,Net sales,"$13,135,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000","Офиц. vs История: $2,505,000,000 (19.1%); Офиц. vs RAW БД: $2,505,000,000 (19.1%); Офиц. vs Canonical: $2,505,000,000 (19.1%)"
1,Net sales to related parties,"$2,505,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000","Офиц. vs История: $13,135,000,000 (524.4%); Офиц. vs RAW БД: $13,135,000,000 (524.4%); Офиц. vs Canonical: $13,135,000,000 (524.4%)"
2,Total net sales,"$15,640,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000",revenue,"$15,640,000,000",✅
3,Cost of sales (excludes items shown below),"$14,060,000,000",cogs,"$14,060,000,000",cogs,"$14,060,000,000",cogs,"$14,060,000,000",✅
4,"Selling, general and administrative expenses","$435,000,000",sga,"$512,000,000",sga,"$435,000,000",sga,"$435,000,000","Офиц. vs История: $77,000,000 (17.7%); История vs RAW БД: $77,000,000"
5,"Depreciation, depletion and amortization","$913,000,000",total_da,"$913,000,000",total_da,"$913,000,000",total_da,"$913,000,000",✅
6,Earnings from investees,"$-112,000,000",-,-,-,-,-,-,✅
7,Asset impairment charges,"$19,000,000",-,-,-,-,-,-,✅
8,Restructuring and other charges,"$8,000,000",-,-,-,-,-,-,✅
9,Gain on equity investee transactions,$0,gain_loss_on_disposal,$0,gain_loss_on_disposal,$0,gain_loss_on_disposal,$0,✅



📋 Сводка сравнения:
  Всего статей в официальном отчете: 27
  Найдено совпадений в RAW БД: 19 (основной источник)
  Найдено совпадений в CANONICAL БД: 18 (после маппинга)
  Найдено совпадений в файле истории: 19 (опционально)
  Статей с расхождениями: 10
  Статей не найденных в БД: 8
  Статей без проблем: 17

📊 Анализ этапов обработки:
  Изменений при загрузке в БД (история → RAW): 7
  Изменений при маппинге (RAW → CANONICAL): 0

  Примеры изменений при загрузке в БД:
    Selling, general and administrative expenses: История=$512,000,000 → RAW=$435,000,000 (diff=$77,000,000)
    Interest income: История=$-426,000,000 → RAW=$-96,000,000 (diff=$330,000,000)
    Net periodic benefit income: История=$-426,000,000 → RAW=$-96,000,000 (diff=$330,000,000)


## 🔴 ШАГ 5.5: Анализ проблем маппинга и создание правильного маппинга

**Цель**: Проанализировать расхождения между официальным отчетом и данными в БД, создать правильный маппинг статей


In [29]:
# === Анализ проблем и создание правильного маппинга ===

display(Markdown("### 🔴 Анализ проблем маппинга"))

if 'comparison_df' not in globals():
    print("❌ Таблица сравнения не создана. Запустите ячейку ШАГ 5.")
else:
    # Анализируем проблемы
    problematic = comparison_df[comparison_df['Проблемы'] != '✅']
    not_found = comparison_df[(comparison_df['Совпадение (RAW БД)'] == '-') & 
                               (comparison_df['Совпадение (canonical)'] == '-')]
    
    print(f"📊 Статистика проблем:")
    print(f"  Статей с расхождениями: {len(problematic)}")
    print(f"  Статей не найденных в БД: {len(not_found)}")
    print(f"  Всего проблемных статей: {len(problematic) + len(not_found)}")
    
    # Группируем проблемы по типам
    print(f"\n🔍 Типы проблем:")
    
    # 1. Неправильный маппинг (значение есть, но не совпадает)
    wrong_mapping = []
    for _, row in problematic.iterrows():
        official = row['Официальный отчет']
        canonical_match = row['Совпадение (canonical)']
        official_val = row['Значение (офиц.)']
        canonical_val = row['Значение (canonical)']
        
        if canonical_match != '-' and canonical_val != '-':
            # Извлекаем числовые значения
            try:
                official_num = float(official_val.replace('$', '').replace(',', ''))
                canonical_num = float(canonical_val.replace('$', '').replace(',', ''))
                diff = abs(official_num - canonical_num)
                diff_pct = (diff / abs(official_num) * 100) if official_num != 0 else 0
                
                if diff_pct > 5:  # Разница больше 5%
                    wrong_mapping.append({
                        'Официальная статья': official,
                        'Офиц. значение': f"${official_num:,.0f}",
                        'Текущий маппинг': canonical_match,
                        'Canonical значение': f"${canonical_num:,.0f}",
                        'Разница': f"${diff:,.0f} ({diff_pct:.1f}%)",
                        'Проблема': 'Неправильный маппинг или значение'
                    })
            except (ValueError, AttributeError):
                pass
    
    if wrong_mapping:
        wrong_mapping_df = pd.DataFrame(wrong_mapping)
        display(Markdown("#### ❌ Неправильный маппинг статей:"))
        display(wrong_mapping_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 2. Статьи не найдены
    if not not_found.empty:
        not_found_display = not_found[['Официальный отчет', 'Значение (офиц.)']].copy()
        not_found_display['Предполагаемый canonical'] = ''
        not_found_display['Комментарий'] = ''
        
        # Предлагаем маппинг для не найденных статей
        mapping_suggestions = {
            'Cash and cash equivalents': 'cash',
            'Receivables, less allowance': 'accounts_receivable',
            'Operating lease assets': 'rou_asset',
            'Property, plant and equipment, net': 'ppe_net',
            'Accounts payable and other accrued liabilities': 'accounts_payable',
            'Short-term debt and current maturities of long-term debt': 'st_debt',
            'Long-term debt, less unamortized discount': 'long_term_debt',
            'Additional paid-in capital': 'apic',
            'Accumulated other comprehensive (loss) income': 'aoci',
            'Total United States Steel Corporation stockholders equity': 'total_equity',
            'Noncontrolling interests': 'nci',
            'Total liabilities and stockholders equity': 'total_liab_equity'
        }
        
        for idx, row in not_found_display.iterrows():
            official_name = row['Официальный отчет']
            if official_name in mapping_suggestions:
                not_found_display.at[idx, 'Предполагаемый canonical'] = mapping_suggestions[official_name]
                not_found_display.at[idx, 'Комментарий'] = 'Предложение маппинга'
            else:
                not_found_display.at[idx, 'Комментарий'] = 'Требуется ручной маппинг'
        
        display(Markdown("#### ⚠️ Статьи не найденные в БД (требуют маппинга):"))
        display(not_found_display.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 3. Создаём правильный маппинг на основе анализа
    print(f"\n📋 Рекомендации по исправлению:")
    print(f"  1. Исправить маппинг для {len(wrong_mapping)} статей с неправильным сопоставлением")
    print(f"  2. Добавить маппинг для {len(not_found)} статей, которые не найдены в БД")
    print(f"  3. Проверить значения в файле истории для статей с расхождениями")
    
    # Сохраняем анализ для дальнейшего использования
    mapping_issues = {
        'wrong_mapping': wrong_mapping,
        'not_found': not_found_display.to_dict('records') if not not_found.empty else []
    }
    
    print(f"\n✅ Анализ проблем сохранен в переменную 'mapping_issues'")
    
    # Создаём правильный маппинг на основе официального отчета
    print(f"\n" + "="*80)
    print("📝 Создание правильного маппинга на основе официального отчета")
    print("="*80)
    
    # Базовый маппинг для основных статей
    correct_mapping = {
        # ASSETS
        'Cash and cash equivalents': 'cash',
        'Receivables, less allowance': 'accounts_receivable',
        'Receivables from related parties': 'receivables_related_parties',
        'Inventories': 'inventory',
        'Other current assets': 'other_current_assets',
        'Total current assets': 'total_current_assets',
        'Long-term restricted cash': 'restricted_cash',
        'Investments and long-term receivables': 'investments_and_long_term_receivables',
        'Operating lease assets': 'rou_asset',
        'Property, plant and equipment, net': 'ppe_net',
        'Intangibles, net': 'intangibles',
        'Deferred income tax benefits': 'dta',  # Deferred Tax Assets
        'Goodwill': 'goodwill',
        'Other noncurrent assets': 'other_non_current_assets',
        'Total assets': 'total_assets',
        
        # LIABILITIES
        'Accounts payable and other accrued liabilities': 'accounts_payable',
        'Accounts payable to related parties': 'accounts_payable_related_parties',  # Новая метрика или часть AP
        'Payroll and benefits payable': 'payroll_and_benefits_payable',
        'Accrued taxes': 'accrued_taxes',  # Новая метрика или часть accrued_liabilities
        'Accrued interest': 'accrued_interest',
        'Current operating lease liabilities': 'lease_liab_current',
        'Short-term debt and current maturities of long-term debt': 'st_debt',
        'Total current liabilities': 'total_current_liabilities',
        'Noncurrent operating lease liabilities': 'lease_liab_noncurrent',
        'Long-term debt, less unamortized discount': 'long_term_debt',
        'Employee benefits': 'employee_benefits',
        'Deferred income tax liabilities': 'dtl',  # Deferred Tax Liabilities
        'Deferred credits and other noncurrent liabilities': 'other_non_current_liabilities',  # Может быть отдельной метрикой
        'Total liabilities': 'total_liabilities',
        
        # EQUITY
        'Common stock issued': 'common_stock_par',
        'Treasury stock, at cost': 'treasury_stock',
        'Additional paid-in capital': 'apic',
        'Retained earnings': 'retained_earnings',
        'Accumulated other comprehensive (loss) income': 'aoci',
        'Total United States Steel Corporation stockholders equity': 'total_equity',
        'Noncontrolling interests': 'nci',
        'Total liabilities and stockholders equity': 'total_liab_equity'
    }
    
    # Проверяем, какие canonical метрики существуют в БД
    existing_canonical = set(canonical_2024.keys())
    
    # Создаём таблицу правильного маппинга
    correct_mapping_list = []
    for official_name, canonical_name in correct_mapping.items():
        official_val = official_reporting_2024.get(official_name)
        canonical_val = canonical_2024.get(canonical_name) if canonical_name in canonical_2024 else None
        
        status = '✅ Найдено' if canonical_name in existing_canonical else '⚠️ Не найдено в БД'
        
        if official_val is not None:
            correct_mapping_list.append({
                'Официальная статья': official_name,
                'Предлагаемый canonical': canonical_name,
                'Офиц. значение': f"${official_val:,.0f}",
                'Текущее значение в БД': f"${canonical_val:,.0f}" if canonical_val is not None else '-',
                'Статус': status
            })
    
    correct_mapping_df = pd.DataFrame(correct_mapping_list)
    display(Markdown("#### 📋 Правильный маппинг статей:"))
    display(correct_mapping_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '8px'}))
    
    # Сохраняем правильный маппинг
    correct_mapping_dict = correct_mapping
    print(f"\n✅ Правильный маппинг создан и сохранен в переменную 'correct_mapping_dict'")
    print(f"   Всего статей в маппинге: {len(correct_mapping_dict)}")
    
    # === Анализ процесса загрузки данных (после создания правильного маппинга) ===
    print(f"\n" + "="*80)
    print("📊 Анализ процесса загрузки данных")
    print("   Официальный отчет → Excel → CSV → RAW БД → CANONICAL БД")
    print("   (маппинг применяется только на этапе RAW → CANONICAL через excel_loader.yaml)")
    print("="*80)
    
    if 'hist_2024_raw' not in globals() or not hist_2024_raw:
        print("⚠️ Файл истории не загружен. Запустите ШАГ 2 для полного анализа.")
    else:
        # Создаём детальное сравнение всех этапов
        loading_analysis = []
        
        for official_item, official_val in official_reporting_2024.items():
            # Используем правильный маппинг для поиска canonical метрики
            canonical_metric_name = correct_mapping_dict.get(official_item)
            
            # Ищем в файле истории (там используются canonical метрики)
            hist_val = None
            hist_match = None
            if canonical_metric_name and canonical_metric_name in hist_2024_raw:
                hist_val = hist_2024_raw[canonical_metric_name]
                hist_match = canonical_metric_name
            else:
                # Fallback: поиск по схожести
                hist_match, hist_score = find_best_match(official_item, hist_2024_raw.keys(), threshold=0.6)
                hist_val = hist_2024_raw.get(hist_match) if hist_match else None
            
            # Ищем в RAW БД (там тоже могут быть canonical метрики или сырые названия)
            raw_val = None
            raw_match = None
            if canonical_metric_name and canonical_metric_name in raw_2024:
                raw_val = raw_2024[canonical_metric_name]
                raw_match = canonical_metric_name
            else:
                # Fallback: поиск по схожести
                raw_match, raw_score = find_best_match(official_item, raw_2024.keys(), threshold=0.6)
                raw_val = raw_2024.get(raw_match) if raw_match else None
            
            # Ищем в CANONICAL БД (там точно canonical метрики)
            canonical_val = None
            canonical_match = None
            if canonical_metric_name and canonical_metric_name in canonical_2024:
                canonical_val = canonical_2024[canonical_metric_name]
                canonical_match = canonical_metric_name
            else:
                # Fallback: поиск по схожести
                canonical_match, canonical_score = find_best_match(official_item, canonical_2024.keys(), threshold=0.6)
                canonical_val = canonical_2024.get(canonical_match) if canonical_match else None
            
            # Анализируем расхождения на каждом этапе
            issues = []
            
            # Этап 1: Официальный → CSV файл истории (из Excel через excel_loader.py)
            # CSV содержит данные как есть из Excel, без маппинга
            if hist_val is not None:
                diff_csv = abs(official_val - hist_val)
                diff_csv_pct = (diff_csv / abs(official_val) * 100) if official_val != 0 else 0
                if diff_csv > 1000:
                    issues.append(f"CSV (Excel→CSV): ${diff_csv:,.0f} ({diff_csv_pct:.1f}%) - проблема в Excel или загрузчике")
            else:
                issues.append("CSV: не найдено - статья отсутствует в Excel/CSV")
            
            # Этап 2: CSV → RAW БД (через load_history_csv_to_db)
            # Должно совпадать, так как загружается как есть
            if hist_val is not None and raw_val is not None:
                diff_raw = abs(hist_val - raw_val)
                if diff_raw > 1000:
                    issues.append(f"RAW (CSV→БД): ${diff_raw:,.0f} - проблема при загрузке CSV в БД")
            elif hist_val is None and raw_val is not None:
                issues.append("RAW: добавлено из другого источника")
            elif hist_val is not None and raw_val is None:
                issues.append("RAW: потеряно при загрузке CSV в БД")
            
            # Этап 3: RAW → CANONICAL БД (через normalize_to_canonical с маппингом из excel_loader.yaml)
            # Здесь применяется маппинг через excel_loader.yaml (секция history.IS.required_metrics)
            if raw_val is not None and canonical_val is not None:
                diff_canonical = abs(raw_val - canonical_val)
                if diff_canonical > 1000:
                    issues.append(f"CANONICAL (маппинг): ${diff_canonical:,.0f} - проблема в маппинге excel_loader.yaml")
            elif raw_val is None and canonical_val is not None:
                issues.append("CANONICAL: добавлено при маппинге (возможно из других источников)")
            elif raw_val is not None and canonical_val is None:
                issues.append("CANONICAL: потеряно при маппинге - нет маппинга в excel_loader.yaml")
            
            # Этап 4: Официальный → CANONICAL (итоговое сравнение)
            if canonical_val is not None:
                diff_final = abs(official_val - canonical_val)
                diff_final_pct = (diff_final / abs(official_val) * 100) if official_val != 0 else 0
                if diff_final > 1000:
                    issues.append(f"ИТОГО: ${diff_final:,.0f} ({diff_final_pct:.1f}%)")
            
            if issues:
                loading_analysis.append({
                    'Официальная статья': official_item,
                    'Canonical метрика': canonical_metric_name or '-',
                    'Офиц. значение': f"${official_val:,.0f}",
                    'CSV файл': f"${hist_val:,.0f}" if hist_val is not None else '-',
                    'RAW БД': f"${raw_val:,.0f}" if raw_val is not None else '-',
                    'CANONICAL БД': f"${canonical_val:,.0f}" if canonical_val is not None else '-',
                    'Проблемы': '; '.join(issues)
                })
        
        if loading_analysis:
            loading_df = pd.DataFrame(loading_analysis)
            display(Markdown("#### 🔍 Анализ процесса загрузки (статьи с расхождениями):"))
            display(loading_df.style.set_table_styles([
                {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
                {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
            ]).set_properties(**{'font-size': '8px'}))
            
            # Группируем проблемы по этапам
            csv_issues = len([x for x in loading_analysis if 'CSV (' in x['Проблемы']])
            raw_issues = len([x for x in loading_analysis if 'RAW (' in x['Проблемы']])
            canonical_issues = len([x for x in loading_analysis if 'CANONICAL (' in x['Проблемы']])
            
            print(f"\n📊 Сводка проблем по этапам:")
            print(f"  Проблемы Excel → CSV (excel_loader.py): {csv_issues} статей")
            print(f"  Проблемы CSV → RAW БД (load_history_csv_to_db): {raw_issues} статей")
            print(f"  Проблемы RAW → CANONICAL (normalize_to_canonical + excel_loader.yaml): {canonical_issues} статей")
            print(f"  Всего проблемных статей: {len(loading_analysis)}")
            
            print(f"\n💡 Рекомендации:")
            if csv_issues > 0:
                print(f"  • Проверьте Excel файл и процесс конвертации через excel_loader.py")
            if raw_issues > 0:
                print(f"  • Проверьте функцию load_history_csv_to_db в engine/database/sqlite_wrapper.py")
            if canonical_issues > 0:
                print(f"  • Проверьте маппинг в excel_loader.yaml (секция history.IS.required_metrics)")
                print(f"  • Проверьте функцию normalize_to_canonical в engine/database/data_mart.py")
            
            # Сохраняем анализ
            loading_analysis_dict = loading_analysis
            print(f"\n✅ Анализ процесса загрузки сохранен в переменную 'loading_analysis_dict'")
        else:
            print("✅ Расхождений в процессе загрузки не обнаружено")


### 🔴 Анализ проблем маппинга

📊 Статистика проблем:
  Статей с расхождениями: 10
  Статей не найденных в БД: 8
  Всего проблемных статей: 18

🔍 Типы проблем:


#### ❌ Неправильный маппинг статей:

,Официальная статья,Офиц. значение,Текущий маппинг,Canonical значение,Разница,Проблема
0,Net sales,"$13,135,000,000",revenue,"$15,640,000,000","$2,505,000,000 (19.1%)",Неправильный маппинг или значение
1,Net sales to related parties,"$2,505,000,000",revenue,"$15,640,000,000","$13,135,000,000 (524.4%)",Неправильный маппинг или значение
2,Net interest and other financial (benefits) costs,"$-198,000,000",interest_income,"$-96,000,000","$102,000,000 (51.5%)",Неправильный маппинг или значение


#### ⚠️ Статьи не найденные в БД (требуют маппинга):

...

#### 📋 Правильный маппинг статей:


✅ Правильный маппинг создан и сохранен в переменную 'correct_mapping_dict'
   Всего статей в маппинге: 37

📊 Анализ процесса загрузки данных
   Официальный отчет → Excel → CSV → RAW БД → CANONICAL БД
   (маппинг применяется только на этапе RAW → CANONICAL через excel_loader.yaml)


#### 🔍 Анализ процесса загрузки (статьи с расхождениями):

,Официальная статья,Canonical метрика,Офиц. значение,CSV файл,RAW БД,CANONICAL БД,Проблемы
0,Net sales,-,"$13,135,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
1,Net sales to related parties,-,"$2,505,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
2,Total net sales,-,"$15,640,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
3,Cost of sales (excludes items shown below),-,"$14,060,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
4,"Selling, general and administrative expenses",-,"$435,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
5,"Depreciation, depletion and amortization",-,"$913,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
6,Earnings from investees,-,"$-112,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
7,Asset impairment charges,-,"$19,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
8,Restructuring and other charges,-,"$8,000,000",-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV
9,Gain on equity investee transactions,-,$0,-,-,-,CSV: не найдено - статья отсутствует в Excel/CSV



📊 Сводка проблем по этапам:
  Проблемы Excel → CSV (excel_loader.py): 3 статей
  Проблемы CSV → RAW БД (load_history_csv_to_db): 2 статей
  Проблемы RAW → CANONICAL (normalize_to_canonical + excel_loader.yaml): 0 статей
  Всего проблемных статей: 25

💡 Рекомендации:
  • Проверьте Excel файл и процесс конвертации через excel_loader.py
  • Проверьте функцию load_history_csv_to_db в engine/database/sqlite_wrapper.py

✅ Анализ процесса загрузки сохранен в переменную 'loading_analysis_dict'


## 📋 ШАГ 6: План исправлений на основе анализа

**Цель**: Создать детальный план исправлений для всех выявленных проблем в процессе загрузки данных


In [ ]:
# === Создание плана исправлений на основе анализа ===

display(Markdown("### 📋 План исправлений процесса загрузки данных"))

if 'loading_analysis_dict' not in globals() or not loading_analysis_dict:
    print("❌ Анализ процесса загрузки не выполнен. Запустите ШАГ 5.5.")
else:
    # Группируем проблемы по типам и приоритетам
    excel_csv_issues = []
    csv_raw_issues = []
    mapping_issues = []
    missing_metrics = []
    
    for item in loading_analysis_dict:
        problems = item['Проблемы']
        official = item['Официальная статья']
        canonical = item['Canonical метрика']
        official_val = item['Офиц. значение']
        csv_val = item['CSV файл']
        raw_val = item['RAW БД']
        canonical_val = item['CANONICAL БД']
        
        # Парсим значения для сравнения
        def parse_value(val_str):
            if val_str == '-' or val_str is None:
                return None
            try:
                return float(val_str.replace('$', '').replace(',', ''))
            except:
                return None
        
        official_num = parse_value(official_val)
        csv_num = parse_value(csv_val)
        raw_num = parse_value(raw_val)
        canonical_num = parse_value(canonical_val)
        
        # Проблемы Excel → CSV
        if 'CSV (Excel→CSV):' in problems:
            excel_csv_issues.append({
                'Статья': official,
                'Canonical': canonical,
                'Офиц. значение': official_val,
                'CSV значение': csv_val,
                'Разница': f"${abs(official_num - csv_num):,.0f}" if official_num and csv_num else '-',
                'Проблема': 'Неправильные данные в Excel или ошибка конвертации excel_loader.py',
                'Действие': 'Проверить Excel файл и excel_loader.py'
            })
        
        # Проблемы CSV → RAW БД
        if 'RAW (CSV→БД):' in problems:
            csv_raw_issues.append({
                'Статья': official,
                'Canonical': canonical,
                'CSV значение': csv_val,
                'RAW значение': raw_val,
                'Разница': f"${abs(csv_num - raw_num):,.0f}" if csv_num and raw_num else '-',
                'Проблема': 'Ошибка при загрузке CSV в БД через load_history_csv_to_db',
                'Действие': 'Проверить функцию load_history_csv_to_db в sqlite_wrapper.py'
            })
        
        # Проблемы маппинга
        if canonical == '-' or canonical_num is None:
            missing_metrics.append({
                'Статья': official,
                'Предлагаемый canonical': correct_mapping_dict.get(official, 'требует определения'),
                'Офиц. значение': official_val,
                'Проблема': 'Отсутствует маппинг в excel_loader.yaml',
                'Действие': 'Добавить маппинг в excel_loader.yaml (секция history.IS.required_metrics)'
            })
    
    # Выводим план исправлений
    print("="*80)
    print("📋 ПЛАН ИСПРАВЛЕНИЙ")
    print("="*80)
    
    # 1. Проблемы Excel → CSV
    if excel_csv_issues:
        print(f"\n🔴 ПРИОРИТЕТ 1: Проблемы Excel → CSV ({len(excel_csv_issues)} статей)")
        print("   Источник проблемы: Excel файл или excel_loader.py")
        print("\n   Действия:")
        print("   1. Проверить Excel файл с историческими данными IS")
        print("   2. Убедиться, что названия метрик в Excel соответствуют canonical метрикам")
        print("   3. Проверить процесс конвертации в excel_loader.py")
        print("   4. Обновить Excel шаблон для правильного маппинга статей")
        
        excel_csv_df = pd.DataFrame(excel_csv_issues)
        display(Markdown("#### 🔴 Проблемы Excel → CSV:"))
        display(excel_csv_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 2. Проблемы CSV → RAW БД
    if csv_raw_issues:
        print(f"\n🟠 ПРИОРИТЕТ 2: Проблемы CSV → RAW БД ({len(csv_raw_issues)} статей)")
        print("   Источник проблемы: функция load_history_csv_to_db")
        print("\n   Действия:")
        print("   1. Проверить функцию load_history_csv_to_db в engine/database/sqlite_wrapper.py")
        print("   2. Убедиться, что данные правильно парсятся из CSV")
        print("   3. Проверить логику определения формата CSV (wide vs long)")
        print("   4. Исправить баги в загрузчике, если они есть")
        
        csv_raw_df = pd.DataFrame(csv_raw_issues)
        display(Markdown("#### 🟠 Проблемы CSV → RAW БД:"))
        display(csv_raw_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # 3. Отсутствующие метрики
    if missing_metrics:
        print(f"\n🟡 ПРИОРИТЕТ 3: Отсутствующие метрики в маппинге ({len(missing_metrics)} статей)")
        print("   Источник проблемы: excel_loader.yaml не содержит маппинг для этих статей")
        print("\n   Действия:")
        print("   1. Открыть companies/us_steel/configs/excel_loader.yaml")
        print("   2. Добавить недостающие метрики в секцию history.IS.required_metrics")
        print("   3. Добавить алиасы для сопоставления названий из Excel с canonical метриками")
        print("   4. Убедиться, что все метрики из официального отчета имеют маппинг")
        
        missing_df = pd.DataFrame(missing_metrics)
        display(Markdown("#### 🟡 Отсутствующие метрики в маппинге:"))
        display(missing_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
    
    # Создаём итоговый план действий
    action_plan = {
        'excel_template': {
            'priority': 1,
            'description': 'Доработать Excel шаблон для загрузки исторических данных IS',
            'actions': [
                'Проверить текущий Excel файл с историческими данными',
                'Убедиться, что названия статей соответствуют canonical метрикам',
                'Добавить недостающие статьи из официального отчета',
                'Исправить неправильные значения (например, Deferred income tax liabilities)',
                'Обновить шаблон для правильного формата данных'
            ],
            'files': [
                'Excel файл с историческими данными IS (нужно найти)',
                'templates/excel_templates/ (если есть шаблон)'
            ]
        },
        'excel_loader': {
            'priority': 1,
            'description': 'Проверить и исправить excel_loader.py',
            'actions': [
                'Проверить функцию load_excel_to_csv',
                'Убедиться, что данные правильно конвертируются из Excel в CSV',
                'Проверить обработку различных форматов Excel',
                'Добавить валидацию данных при конвертации'
            ],
            'files': [
                'tools/excel_loader.py'
            ]
        },
        'csv_to_db': {
            'priority': 2,
            'description': 'Исправить загрузку CSV в БД',
            'actions': [
                'Проверить функцию load_history_csv_to_db в sqlite_wrapper.py',
                'Убедиться, что данные правильно парсятся из CSV',
                'Проверить логику определения формата CSV (wide vs long)',
                'Исправить баги, которые приводят к неправильным значениям',
                'Добавить логирование для отладки'
            ],
            'files': [
                'engine/database/sqlite_wrapper.py (функция load_history_csv_to_db)'
            ]
        },
        'mapping_yaml': {
            'priority': 3,
            'description': 'Доработать маппинг в excel_loader.yaml',
            'actions': [
                'Добавить недостающие метрики в history.IS.required_metrics',
                'Добавить алиасы для сопоставления названий из Excel',
                'Убедиться, что все статьи из официального отчета имеют маппинг',
                'Проверить правильность существующих маппингов'
            ],
            'files': [
                'companies/us_steel/configs/excel_loader.yaml'
            ]
        }
    }
    
    print(f"\n" + "="*80)
    print("📝 ИТОГОВЫЙ ПЛАН ДЕЙСТВИЙ")
    print("="*80)
    
    for key, plan in sorted(action_plan.items(), key=lambda x: x[1]['priority']):
        print(f"\n{'🔴' if plan['priority'] == 1 else '🟠' if plan['priority'] == 2 else '🟡'} ПРИОРИТЕТ {plan['priority']}: {plan['description']}")
        print(f"   Файлы для изменения:")
        for file in plan['files']:
            print(f"     • {file}")
        print(f"   Действия:")
        for i, action in enumerate(plan['actions'], 1):
            print(f"     {i}. {action}")
    
    # Сохраняем план
    action_plan_dict = action_plan
    print(f"\n✅ План действий сохранен в переменную 'action_plan_dict'")
    
    # Создаём список конкретных исправлений для excel_loader.yaml
    print(f"\n" + "="*80)
    print("📝 КОНКРЕТНЫЕ ИСПРАВЛЕНИЯ ДЛЯ excel_loader.yaml")
    print("="*80)
    
    yaml_fixes = []
    for item in loading_analysis_dict:
        official = item['Официальная статья']
        canonical = correct_mapping_dict.get(official)
        
        if canonical and canonical not in ['-', None]:
            # Проверяем, есть ли эта метрика в required_metrics
            if required_metrics and canonical not in required_metrics:
                yaml_fixes.append({
                    'Canonical метрика': canonical,
                    'Официальная статья': official,
                    'Действие': f'Добавить в history.IS.required_metrics',
                    'Пример YAML': f"      {canonical}:\n        aliases:\n        - {official.lower().replace(' ', '_')}\n        required_for:\n        - standard\n        - custom"
                })
    
    if yaml_fixes:
        yaml_fixes_df = pd.DataFrame(yaml_fixes)
        display(Markdown("#### 📝 Недостающие метрики для добавления в excel_loader.yaml:"))
        display(yaml_fixes_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
        
        print(f"\n💡 Пример обновления excel_loader.yaml:")
        print(f"   Добавьте следующие метрики в секцию history.IS.required_metrics:")
        for fix in yaml_fixes[:5]:  # Показываем первые 5
            print(f"\n{fix['Пример YAML']}")
        
        if len(yaml_fixes) > 5:
            print(f"\n   ... и еще {len(yaml_fixes) - 5} метрик")
    
    print(f"\n✅ Анализ завершен. Используйте план действий для исправления проблем.")


## 🔧 ШАГ 7: Создание правильного canonical формата

In [ ]:
# === Создание правильного canonical формата на основе официальной отчетности ===

display(Markdown("### 🔧 Создание правильного canonical формата для 2024 года"))

if not official_reporting_2024:
    print("❌ Данные из официальной отчетности не загружены. Запустите ячейку ШАГ 1.")
elif not canonical_2024:
    print("❌ Данные из canonical формата не загружены. Запустите ячейку ШАГ 3.")
else:
    # Создаём маппинг алиасов
    aliases_map = {}
    for metric, config in required_metrics.items():
        aliases = config.get('aliases', [])
        for alias in aliases:
            aliases_map[alias.lower()] = metric
        aliases_map[metric.lower()] = metric
    
    # Используем правильный маппинг из анализа проблем (ШАГ 5.5), если он создан
    if 'correct_mapping_dict' in globals() and correct_mapping_dict:
        print("✅ Используем правильный маппинг из анализа проблем (ШАГ 5.5)")
        print(f"   Загружено {len(correct_mapping_dict)} правильных сопоставлений")
        mapping_to_use = correct_mapping_dict
    else:
        print("⚠️ Правильный маппинг не найден, используем автоматический поиск")
        print("   Рекомендуется сначала запустить ШАГ 5.5 для анализа проблем")
    
    # Создаём функцию для поиска canonical метрики с использованием той же логики, что и в ШАГ 5
    def find_canonical_metric_for_step7(official_name):
        """Находит canonical метрику используя ту же логику, что и в ШАГ 5"""
        # ПРИОРИТЕТ 1: Используем явный маппинг официальных названий (из ШАГ 5)
        if 'official_to_canonical_map' in globals() and official_name in official_to_canonical_map:
            canonical_name = official_to_canonical_map[official_name]
            if canonical_name in canonical_2024:
                return canonical_name, 'official_to_canonical_map'
        
        # ПРИОРИТЕТ 2: Используем правильный маппинг из анализа проблем (ШАГ 5.5)
        if 'correct_mapping_dict' in globals() and correct_mapping_dict and official_name in correct_mapping_dict:
            canonical_name = correct_mapping_dict[official_name]
            if canonical_name and canonical_name in canonical_2024:
                return canonical_name, 'correct_mapping_dict'
        
        # ПРИОРИТЕТ 3: Ищем через алиасы (нормализуем название)
        official_normalized_tuple = normalize_name(official_name)
        official_normalized = official_normalized_tuple[0] if isinstance(official_normalized_tuple, tuple) else official_normalized_tuple
        official_normalized_no_spaces = official_normalized_tuple[1] if isinstance(official_normalized_tuple, tuple) else official_normalized.replace(' ', '')
        
        if official_normalized in aliases_map:
            canonical_name = aliases_map[official_normalized]
            if canonical_name in canonical_2024:
                return canonical_name, 'aliases_map'
        elif official_normalized_no_spaces in aliases_map:
            canonical_name = aliases_map[official_normalized_no_spaces]
            if canonical_name in canonical_2024:
                return canonical_name, 'aliases_map'
        else:
            # Проверяем частичное совпадение с алиасами
            for alias, canonical in aliases_map.items():
                alias_normalized = normalize_name_simple(alias).replace(' ', '')
                if (official_normalized in alias or alias in official_normalized or
                    official_normalized_no_spaces in alias_normalized or alias_normalized in official_normalized_no_spaces):
                    if canonical in canonical_2024:
                        return canonical, 'aliases_map_partial'
        
        # ПРИОРИТЕТ 4: Поиск через canonical_to_aliases (из ШАГ 5)
        if 'canonical_to_aliases' in globals():
            for canonical_metric, aliases_list in canonical_to_aliases.items():
                for alias in aliases_list:
                    alias_normalized_tuple = normalize_name(alias)
                    alias_normalized = alias_normalized_tuple[0] if isinstance(alias_normalized_tuple, tuple) else alias_normalized_tuple
                    alias_normalized_no_spaces = alias_normalized_tuple[1] if isinstance(alias_normalized_tuple, tuple) else alias_normalized.replace(' ', '')
                    if (official_normalized == alias_normalized or 
                        alias_normalized in official_normalized or
                        official_normalized_no_spaces == alias_normalized_no_spaces or
                        alias_normalized_no_spaces in official_normalized_no_spaces):
                        if canonical_metric in canonical_2024:
                            return canonical_metric, 'canonical_to_aliases'
        
        # ПРИОРИТЕТ 5: По схожести с canonical метриками
        canonical_match, score = find_best_match(official_name, canonical_2024.keys(), threshold=0.6)
        if canonical_match:
            return canonical_match, f'similarity ({score:.1%})'
        
        # ПРИОРИТЕТ 6: Через историю (если доступна)
        if 'hist_2024_raw' in globals() and hist_2024_raw:
            hist_match, hist_score = find_best_match(official_name, hist_2024_raw.keys(), threshold=0.6)
            if hist_match and hist_match in canonical_2024:
                return hist_match, f'history ({hist_score:.1%})'
        
        return None, None
    
    # Создаём правильный canonical формат
    corrected_canonical = {}
    mapping_log = []
    
    print("\n📋 Создание маппинга официальных статей на canonical метрики:")
    
    # Словарь для отслеживания статей, которые должны суммироваться
    articles_to_sum = {}  # {canonical_metric: [list of official items]}
    
    for official_item, official_val in official_reporting_2024.items():
        # Используем многоуровневую систему поиска (та же логика, что и в ШАГ 5)
        canonical_metric, method = find_canonical_metric_for_step7(official_item)
        
        if canonical_metric:
            # ВАЖНО: Исправляем неправильные маппинги перед суммированием
            # 1. "Other losses (gains), net" должна быть в other_expenses, а не в sga
            if 'other losses (gains)' in official_item.lower() and canonical_metric == 'sga':
                canonical_metric = 'other_expenses'
                method = 'corrected_mapping'
            # 2. "Loss on debt extinguishment" должна быть в other_expenses, а не в ebt
            elif 'loss on debt extinguishment' in official_item.lower() and canonical_metric == 'ebt':
                canonical_metric = 'other_expenses'
                method = 'corrected_mapping'
            # 3. "Net periodic benefit income" должна быть в other_income, а не в interest_income
            elif 'net periodic benefit income' in official_item.lower() and canonical_metric == 'interest_income':
                canonical_metric = 'other_income'
                method = 'corrected_mapping'
            # 4. "Net interest and other financial (benefits) costs" - это агрегированная статья, пропускаем её
            elif 'net interest and other financial' in official_item.lower():
                print(f"⚠️ Статья '{official_item}' пропущена (агрегированная статья, нельзя суммировать)")
                continue
            # 5. "Total operating expenses" - это агрегированная статья, пропускаем её
            elif 'total operating expenses' in official_item.lower():
                print(f"⚠️ Статья '{official_item}' пропущена (агрегированная статья, нельзя суммировать)")
                continue
            # 6. "Net earnings attributable to United States Steel Corporation" - это дубликат "Net earnings", пропускаем
            elif 'net earnings attributable to united states steel corporation' in official_item.lower():
                print(f"⚠️ Статья '{official_item}' пропущена (дубликат 'Net earnings')")
                continue
            
            # ВАЖНО: Если несколько статей маппятся в одну canonical метрику, они должны СУММИРОВАТЬСЯ
            # Например: "Net sales" + "Net sales to related parties" → revenue
            if canonical_metric not in articles_to_sum:
                articles_to_sum[canonical_metric] = []
            articles_to_sum[canonical_metric].append({
                'official_item': official_item,
                'value': official_val,
                'method': method
            })
            
            # Проверяем, есть ли эта метрика в текущем canonical формате
            current_canonical_val = canonical_2024.get(canonical_metric)
            status_icon = '✅' if current_canonical_val is not None else '🆕'
            status_text = 'обновлено' if current_canonical_val is not None else 'новая метрика'
            
            # НЕ добавляем в mapping_log здесь, добавим после суммирования
        else:
            # Статья не маппится - должна попасть в "прочее"
            # ВАЖНО: Правильно определяем тип статьи для "прочего"
            
            # Определяем тип: доход или расход на основе названия и значения
            item_lower = official_item.lower()
            is_income = any(keyword in item_lower for keyword in [
                "income", "gain", "earnings from", "net gain", "benefit income"
            ])
            is_expense = any(keyword in item_lower for keyword in [
                "expense", "loss", "charge", "cost", "impairment", "restructuring", 
                "other losses", "other financial costs", "debt extinguishment"
            ])
            
            # Специальные случаи
            if "other losses (gains)" in item_lower or "other losses" in item_lower:
                # "Other losses (gains), net" - это расход (если положительное значение)
                is_expense = True
                is_income = False
            elif "net interest and other financial" in item_lower:
                # "Net interest and other financial (benefits) costs" - это агрегированная метрика
                # Не должна попадать в other_income, т.к. это уже сумма interest + other
                # Но если она не маппится, значит нужно разобрать на компоненты или добавить в other_expenses
                is_expense = True
                is_income = False
            elif "net periodic benefit income" in item_lower:
                # Это доход (benefit income)
                is_income = True
                is_expense = False
            
            # Определяем canonical метрику для "прочего"
            if is_income:
                other_metric = "other_income"
            elif is_expense:
                other_metric = "other_expenses"
            else:
                # Если не можем определить, используем знак значения
                other_metric = "other_income" if official_val < 0 else "other_expenses"  # Отрицательное = доход (для некоторых статей)
            
            # Добавляем в суммирование для "прочего"
            if other_metric not in articles_to_sum:
                articles_to_sum[other_metric] = []
            articles_to_sum[other_metric].append({
                'official_item': official_item,
                'value': official_val,
                'method': 'other_items'
            })
            
            print(f"⚠️ Статья '{official_item}' не сопоставлена, будет добавлена в '{other_metric}'")
    
    # Теперь суммируем все статьи для каждой canonical метрики
    print("\n📊 Суммирование статей для canonical метрик:")
    for canonical_metric, articles_list in articles_to_sum.items():
        if len(articles_list) > 1:
            # ВАЖНО: Проверяем, есть ли среди статей "Total" или агрегированная статья
            # Если есть, используем её вместо суммирования компонентов
            total_articles = [art for art in articles_list if 'total' in art['official_item'].lower()]
            
            # ВАЖНО: Исключаем агрегированные статьи, которые нельзя суммировать
            # 1. "Total operating expenses" - это агрегированная статья, её нельзя суммировать с отдельными статьями
            # 2. "Net interest and other financial (benefits) costs" - это агрегированная статья
            # 3. "Net earnings attributable to United States Steel Corporation" - это та же статья, что и "Net earnings"
            aggregated_articles = [
                'total operating expenses',
                'net interest and other financial (benefits) costs',
                'net earnings attributable to united states steel corporation'
            ]
            
            # Фильтруем агрегированные статьи
            filtered_articles = []
            excluded_aggregated = []
            for art in articles_list:
                art_lower = art['official_item'].lower()
                is_aggregated = any(agg in art_lower for agg in aggregated_articles)
                if is_aggregated:
                    excluded_aggregated.append(art)
                else:
                    filtered_articles.append(art)
            
            if excluded_aggregated:
                print(f"     ⚠️ Исключены агрегированные статьи (нельзя суммировать с компонентами):")
                for art in excluded_aggregated:
                    print(f"       - {art['official_item']}: ${art['value']:,.0f}")
            
            # Используем отфильтрованный список
            articles_list = filtered_articles
            
            if total_articles and len(articles_list) > 1:
                # ВАЖНО: Используем агрегированную статью (например, "Total net sales")
                # ИСКЛЮЧАЕМ компоненты из суммирования, т.к. они уже учтены в Total
                total_art = total_articles[0]
                total_value = total_art['value']
                
                # Определяем компоненты, которые нужно исключить
                component_keywords = []
                if 'total net sales' in total_art['official_item'].lower():
                    component_keywords = ['net sales', 'sales to related']
                elif 'net earnings' in total_art['official_item'].lower() and 'attributable' in total_art['official_item'].lower():
                    component_keywords = ['net earnings']  # Исключаем просто "Net earnings"
                
                excluded_articles = []
                included_articles = [total_art]
                
                for art in articles_list:
                    if art != total_art:
                        # Проверяем, является ли статья компонентом агрегированной
                        is_component = any(keyword in art['official_item'].lower() for keyword in component_keywords)
                        if is_component:
                            excluded_articles.append(art)
                        else:
                            # Если статья не является компонентом, но маппится в ту же метрику - это ошибка маппинга
                            print(f"     ⚠️ {art['official_item']}: ${art['value']:,.0f} - возможно неправильный маппинг (не компонент Total)")
                            included_articles.append(art)
                
                # Если есть исключенные компоненты, используем только Total
                if excluded_articles:
                    print(f"  ✅ {canonical_metric}: используется агрегированная статья '{total_art['official_item']}' = ${total_value:,.0f}")
                    print(f"     (исключены компоненты из суммирования, т.к. 'Total' уже содержит сумму):")
                    for art in excluded_articles:
                        print(f"       - {art['official_item']}: ${art['value']:,.0f} (компонент, учтен в Total)")
                    
                    # Добавляем только агрегированную статью в mapping_log
                    mapping_log.append({
                        'Официальное название': total_art['official_item'],
                        'Canonical метрика': canonical_metric,
                        'Метод': total_art['method'],
                        'Офиц. значение': f"${total_art['value']:,.0f}",
                        'Текущее в БД': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_2024 else '-',
                        'Статус': 'используется агрегированная'
                    })
                else:
                    # Нет компонентов для исключения - суммируем все статьи
                    total_value = sum(art['value'] for art in articles_list)
                    print(f"  ✅ {canonical_metric}: суммируется из {len(articles_list)} статей = ${total_value:,.0f}")
                    for art in articles_list:
                        print(f"     - {art['official_item']}: ${art['value']:,.0f} (via {art['method']})")
                        mapping_log.append({
                            'Официальное название': art['official_item'],
                            'Canonical метрика': canonical_metric,
                            'Метод': art['method'],
                            'Офиц. значение': f"${art['value']:,.0f}",
                            'Текущее в БД': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_2024 else '-',
                            'Статус': 'суммируется'
                        })
            else:
                # Несколько статей суммируются (нет агрегированной)
                total_value = sum(art['value'] for art in articles_list)
                print(f"  ✅ {canonical_metric}: суммируется из {len(articles_list)} статей = ${total_value:,.0f}")
                for art in articles_list:
                    print(f"     - {art['official_item']}: ${art['value']:,.0f} (via {art['method']})")
                    mapping_log.append({
                        'Официальное название': art['official_item'],
                        'Canonical метрика': canonical_metric,
                        'Метод': art['method'],
                        'Офиц. значение': f"${art['value']:,.0f}",
                        'Текущее в БД': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_2024 else '-',
                        'Статус': 'суммируется'
                    })
            
            corrected_canonical[canonical_metric] = total_value
        else:
            # Одна статья - просто присваиваем значение
            art = articles_list[0]
            corrected_canonical[canonical_metric] = art['value']
            mapping_log.append({
                'Официальное название': art['official_item'],
                'Canonical метрика': canonical_metric,
                'Метод': art['method'],
                'Офиц. значение': f"${art['value']:,.0f}",
                'Текущее в БД': f"${canonical_2024.get(canonical_metric, 0):,.0f}" if canonical_2024 else '-',
                'Статус': 'обновлено' if canonical_metric in canonical_2024 else 'новая метрика'
            })
        
        # Проверяем, есть ли эта метрика в текущем canonical формате
        current_canonical_val = canonical_2024.get(canonical_metric)
        if current_canonical_val is not None:
            final_value = corrected_canonical[canonical_metric]
            diff = final_value - current_canonical_val
            if abs(diff) > 1000:  # Разница больше $1K
                status_icon = '✅'
                status_text = 'обновлено'
                print(f"{status_icon} {canonical_metric}: ${final_value:,.0f} (сумма из {len(articles_list)} статей) - {status_text} (было ${current_canonical_val:,.0f}, diff=${diff:,.0f})")
            else:
                print(f"✅ {canonical_metric}: ${final_value:,.0f} - совпадает")
        else:
            print(f"🆕 {canonical_metric}: ${corrected_canonical[canonical_metric]:,.0f} - новая метрика")
    
    # Добавляем статьи, которые есть в canonical, но не в официальном отчете (оставляем как есть)
    for canonical_metric, canonical_val in canonical_2024.items():
        if canonical_metric not in corrected_canonical:
            corrected_canonical[canonical_metric] = canonical_val
            print(f"ℹ️ {canonical_metric}: ${canonical_val:,.0f} (сохранено из canonical)")
    
    print(f"\n✅ Создан исправленный canonical формат: {len(corrected_canonical)} статей")
    
    # Показываем маппинг
    mapping_df = pd.DataFrame(mapping_log)
    display(Markdown("#### 📋 Таблица маппинга:"))
    display(mapping_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
        {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
    ]).set_properties(**{'font-size': '8px'}))
    
    # Показываем расхождения
    print("\n" + "="*80)
    print("Расхождения между оригинальным и исправленным canonical:")
    
    differences = []
    for metric in sorted(set(canonical_2024.keys()) | set(corrected_canonical.keys())):
        orig_val = canonical_2024.get(metric, 0)
        corr_val = corrected_canonical.get(metric, 0)
        
        if abs(orig_val - corr_val) > 1000:  # Разница больше $1K
            diff = corr_val - orig_val
            diff_pct = (diff / abs(orig_val) * 100) if orig_val != 0 else 0
            differences.append({
                'Метрика': metric,
                'Оригинал (canonical)': f"${orig_val:,.0f}",
                'Исправлено (офиц.)': f"${corr_val:,.0f}",
                'Разница': f"${diff:,.0f}",
                'Разница %': f"{diff_pct:+.1f}%"
            })
    
    if differences:
        diff_df = pd.DataFrame(differences)
        display(Markdown("#### ❌ Статьи с расхождениями:"))
        display(diff_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '9px'), ('padding', '3px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '8px'), ('padding', '2px'), ('text-align', 'left')]}
        ]).set_properties(**{'font-size': '8px'}))
        print(f"\n⚠️ Обнаружено {len(differences)} статей с расхождениями > $1K")
    else:
        print("✅ Расхождений не найдено")
    
    # Сохраняем в переменную
    corrected_canonical_2024 = corrected_canonical
    print("\n✅ Исправленный canonical формат сохранен в переменную 'corrected_canonical_2024'")

## 🔧 ШАГ 7.1: Применение исправлений

**Цель**: Применить все исправления на основе правильного canonical формата:
1. Обновить CSV файл истории для 2024 года
2. Обновить excel_loader.yaml для добавления недостающих метрик
3. Обновить данные в БД


In [ ]:
# === Применение исправлений ===

display(Markdown("### 🔧 Применение исправлений на основе правильного canonical формата"))

if 'corrected_canonical_2024' not in globals() or not corrected_canonical_2024:
    print("❌ Исправленный canonical формат не создан. Запустите ШАГ 7.")
else:
    csv_updated = False
    yaml_updated = False
    db_updated = False
    print("="*80)
    print("🔧 ПРИМЕНЕНИЕ ИСПРАВЛЕНИЙ")
    print("="*80)
    
    # 1. Обновление CSV файла истории
    print(f"\n📝 ШАГ 1: Обновление CSV файла истории для 2024 года")
    
    csv_path = croot / "history" / f"is_history_{COMPANY}.csv"
    
    if not csv_path.exists():
        print(f"❌ CSV файл не найден: {csv_path}")
        csv_updated = False
    else:
        # Читаем текущий CSV
        df_csv = pd.read_csv(csv_path)
        
        # Обновляем значения для 2024 года
        updates_count = 0
        new_metrics_count = 0
        
        # ВАЖНО: corrected_canonical_2024 уже содержит правильные суммированные значения из ШАГ 7
        # Применяем их как есть, без дополнительного суммирования
        for metric, value in corrected_canonical_2024.items():
            if metric in df_csv['metric'].values:
                # Обновляем существующую метрику значением из corrected_canonical_2024 (уже суммировано)
                idx = df_csv[df_csv['metric'] == metric].index[0]
                old_value = df_csv.at[idx, '2024']
                df_csv.at[idx, '2024'] = value
                if abs(old_value - value) > 1000:
                    updates_count += 1
                    print(f"   ✅ Обновлено {metric}: ${old_value:,.0f} → ${value:,.0f}")
            else:
                # Добавляем новую метрику
                new_row = {'metric': metric}
                # Заполняем нулями для всех годов кроме 2024
                for col in df_csv.columns:
                    if col == 'metric':
                        continue
                    new_row[col] = 0.0
                new_row['2024'] = value
                df_csv = pd.concat([df_csv, pd.DataFrame([new_row])], ignore_index=True)
                new_metrics_count += 1
                print(f"   🆕 Добавлено {metric}: ${value:,.0f}")
        
        # Сохраняем обновленный CSV
        df_csv.to_csv(csv_path, index=False)
        print(f"\n✅ CSV файл обновлен:")
        print(f"   Обновлено метрик: {updates_count}")
        print(f"   Добавлено новых метрик: {new_metrics_count}")
        print(f"   Путь: {csv_path}")
        
        csv_updated = True
    
    # 2. Обновление excel_loader.yaml
    print(f"\n📝 ШАГ 2: Обновление excel_loader.yaml для добавления недостающих метрик")
    
    yaml_path = croot / "configs" / "excel_loader.yaml"
    
    if not yaml_path.exists():
        print(f"❌ YAML файл не найден: {yaml_path}")
        yaml_updated = False
    else:
        # Читаем текущий YAML
        with open(yaml_path, 'r', encoding='utf-8') as f:
            yaml_config = yaml.safe_load(f)
        
        # Получаем секцию IS required_metrics
        is_required = yaml_config.get('history', {}).get('IS', {}).get('required_metrics', {})
        
        # Определяем недостающие метрики
        missing_metrics = []
        for metric in corrected_canonical_2024.keys():
            if metric not in is_required:
                # Находим официальное название для создания алиаса
                official_name = None
                if 'correct_mapping_dict' in globals():
                    for official, canonical in correct_mapping_dict.items():
                        if canonical == metric:
                            official_name = official
                            break
                
                missing_metrics.append({
                    'metric': metric,
                    'official_name': official_name
                })
        
        if missing_metrics:
            print(f"   Найдено недостающих метрик: {len(missing_metrics)}")
            
            # Добавляем недостающие метрики
            for item in missing_metrics:
                metric = item['metric']
                official_name = item['official_name']
                
                # Создаем структуру для новой метрики
                metric_config = {
                    'required_for': ['standard', 'custom']
                }
                
                # Добавляем алиасы если есть официальное название
                if official_name:
                    # Создаем алиас из официального названия
                    alias = official_name.lower().replace(' ', '_').replace(',', '').replace('(', '').replace(')', '')
                    metric_config['aliases'] = [alias]
                
                is_required[metric] = metric_config
                print(f"   ✅ Добавлено: {metric}" + (f" (алиас: {alias})" if official_name else ""))
            
            # Сохраняем обновленный YAML
            with open(yaml_path, 'w', encoding='utf-8') as f:
                yaml.dump(yaml_config, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
            
            print(f"\n✅ excel_loader.yaml обновлен:")
            print(f"   Добавлено метрик: {len(missing_metrics)}")
            print(f"   Путь: {yaml_path}")
            
            yaml_updated = True
        else:
            print(f"   ✅ Все метрики уже присутствуют в excel_loader.yaml")
            yaml_updated = False
    
    # 3. Обновление данных в БД
    print(f"\n📝 ШАГ 3: Обновление данных в БД")
    
    try:
        mart = get_data_mart(ROOT, COMPANY)
        if mart is None:
            print("❌ Не удалось подключиться к витрине данных")
        else:
            # Обновляем данные для 2024 года
            updates_db = []
            
            # ВАЖНО: corrected_canonical_2024 уже содержит правильные суммированные значения из ШАГ 7
            # Применяем их как есть, без дополнительного суммирования
            for metric, value in corrected_canonical_2024.items():
                try:
                    # Используем прямой SQL для обновления
                    cursor = mart.db.conn.cursor()
                    
                    # Проверяем, существует ли запись
                    cursor.execute("""
                        SELECT value FROM history_is 
                        WHERE company = ? AND metric = ? AND year = ?
                    """, (COMPANY, metric, 2024))
                    
                    existing = cursor.fetchone()
                    
                    if existing:
                        # Обновляем существующую запись значением из corrected_canonical_2024 (уже суммировано)
                        cursor.execute("""
                            UPDATE history_is 
                            SET value = ? 
                            WHERE company = ? AND metric = ? AND year = ?
                        """, (value, COMPANY, metric, 2024))
                        updates_db.append({'metric': metric, 'action': 'обновлено', 'value': value})
                    else:
                        # Вставляем новую запись
                        cursor.execute("""
                            INSERT INTO history_is (company, metric, year, value)
                            VALUES (?, ?, ?, ?)
                        """, (COMPANY, metric, 2024, value))
                        updates_db.append({'metric': metric, 'action': 'добавлено', 'value': value})
                    
                    mart.db.conn.commit()
                    
                except Exception as e:
                    print(f"   ⚠️ Ошибка при обновлении {metric}: {e}")
                    mart.db.conn.rollback()
            
            print(f"\n✅ Данные в БД обновлены:")
            print(f"   Всего метрик обработано: {len(updates_db)}")
            
            # Показываем статистику
            updated_count = len([x for x in updates_db if x['action'] == 'обновлено'])
            added_count = len([x for x in updates_db if x['action'] == 'добавлено'])
            print(f"   Обновлено: {updated_count}")
            print(f"   Добавлено: {added_count}")
            
            db_updated = True
            mart.close()
            
    except Exception as e:
        print(f"❌ Ошибка при обновлении БД: {e}")
        import traceback
        traceback.print_exc()
        db_updated = False
    
    # Итоговая сводка
    print(f"\n" + "="*80)
    print("✅ ИТОГОВАЯ СВОДКА ИСПРАВЛЕНИЙ")
    print("="*80)
    print(f"   CSV файл истории: {'✅ Обновлен' if csv_updated else '❌ Не обновлен'}")
    print(f"   excel_loader.yaml: {'✅ Обновлен' if yaml_updated else 'ℹ️ Не требовал обновления'}")
    print(f"   База данных: {'✅ Обновлена' if db_updated else '❌ Не обновлена'}")
    
    if csv_updated and db_updated:
        print(f"\n💡 Следующие шаги:")
        print(f"   1. Выполните ШАГ 7.2 для экспорта данных из БД в CSV файлы истории")
        print(f"   2. Выполните ШАГ 7.3 для обновления файлов Edgar")
        print(f"   3. Загрузите всю историю через load_edgar_to_data_mart.py")
        print(f"   4. Перезапустите препроцессинг для применения изменений")
        print(f"   5. Проверьте данные через ШАГ 9 (Проверка исправленных данных)")
        print(f"   6. Убедитесь, что отчет о прибылях и убытках сходится (Revenue - Expenses = Net Income)")
    
    print(f"\n✅ Исправления применены!")


## 🔄 ШАГ 7.2: Экспорт данных из БД в CSV файлы истории

**Цель**: Синхронизировать CSV файлы истории с обновленными данными из БД

**Важно**: 
- После обновления данных в БД через ШАГ 7.1, нужно экспортировать их обратно в CSV файлы истории
- Это обеспечит синхронизацию между БД и CSV файлами


In [ ]:
# === Экспорт данных из БД в CSV файлы истории ===

display(Markdown("### 🔄 Экспорт данных из БД в CSV файлы истории"))

print("="*80)
print("ЭКСПОРТ ИСТОРИИ ИЗ БД В CSV")
print("="*80)

try:
    mart = get_data_mart(ROOT, COMPANY)
    if mart is None:
        print("❌ Не удалось подключиться к витрине данных")
    else:
        # Экспортируем историю в CSV файлы
        history_dir = croot / "history"
        history_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"\n📁 Директория для экспорта: {history_dir}")
        
        # Экспортируем IS, BS, CF
        for statement_type in ['IS', 'BS', 'CF']:
            try:
                # Загружаем данные из БД
                if statement_type == 'IS':
                    df = mart.get_history_income_statement(canonical=False)
                elif statement_type == 'BS':
                    df = mart.get_history_income_statement(canonical=False)
                else:  # CF
                    df = mart.get_history_cash_flow(canonical=False)
                
                if df.empty:
                    print(f"  ⚠️ {statement_type}: нет данных в БД")
                    continue
                
                # Преобразуем в wide формат (metric, 2006, 2007, ..., 2024)
                # Убираем колонки company, source, unit если есть
                cols_to_drop = ['company', 'source', 'unit']
                for col in cols_to_drop:
                    if col in df.columns:
                        df = df.drop(columns=[col])
                
                # Pivot: metric как индекс, year как колонки
                if 'year' in df.columns and 'value' in df.columns:
                    df_wide = df.pivot_table(
                        index='metric',
                        columns='year',
                        values='value',
                        aggfunc='first'
                    ).reset_index()
                    df_wide.columns.name = None
                else:
                    # Уже в wide формате
                    df_wide = df.copy()
                
                # Сохраняем в правильный формат: is_history_us_steel.csv
                csv_filename = f"{statement_type.lower()}_history_{COMPANY}.csv"
                csv_path = history_dir / csv_filename
                df_wide.to_csv(csv_path, index=False)
                
                print(f"  ✅ {statement_type}: экспортировано {len(df_wide)} метрик → {csv_filename}")
                
            except Exception as e:
                print(f"  ❌ Ошибка при экспорте {statement_type}: {e}")
                import traceback
                traceback.print_exc()
        
        mart.close()
        
        print(f"\n✅ Экспорт завершен!")
        print(f"   CSV файлы обновлены в директории: {history_dir}")
        print(f"   Теперь CSV файлы синхронизированы с БД")
        
except Exception as e:
    print(f"❌ Ошибка при экспорте: {e}")
    import traceback
    traceback.print_exc()


## 🔄 ШАГ 7.3: Загрузка всей истории из Edgar в БД

**Цель**: Загрузить всю историю IS из файлов Edgar в БД через `load_edgar_to_data_mart.py` с правильным маппингом

**Важно**: 
- **НЕ обновляем файлы Edgar** - они остаются как исходные данные из SEC
- Исправления применяются через **маппинг** в `excel_loader.yaml` (уже обновлен в ШАГ 7.1)
- При загрузке через `load_edgar_to_data_mart.py` маппинг RAW → CANONICAL применяется автоматически
- Данные для 2024 года уже исправлены в БД через ШАГ 7.1
- Загрузка из Edgar перезагрузит всю историю с правильным маппингом


In [ ]:
# === Загрузка всей истории из Edgar в БД ===

display(Markdown("### 🔄 Загрузка всей истории из Edgar в БД"))

print("="*80)
print("ЗАГРУЗКА ВСЕЙ ИСТОРИИ ИЗ EDGAR В БД")
print("="*80)

print("\n💡 ВАЖНО:")
print("   • Файлы Edgar НЕ обновляются - они остаются как исходные данные из SEC")
print("   • Исправления применяются через маппинг в excel_loader.yaml (уже обновлен в ШАГ 7.1)")
print("   • При загрузке через load_edgar_to_data_mart.py маппинг RAW → CANONICAL применяется автоматически")
print("   • Данные для 2024 года уже исправлены в БД через ШАГ 7.1")
print("   • Загрузка из Edgar перезагрузит всю историю с правильным маппингом")

if 'corrected_canonical_2024' not in globals() or not corrected_canonical_2024:
    print("\n❌ Исправленный canonical формат не создан. Запустите ШАГ 7.")
    print("   ШАГ 7 создаст переменную corrected_canonical_2024 с правильными значениями")
else:
    # Путь к директории Edgar - автоматический поиск
    import os
    from pathlib import Path
    
    # Приоритетная директория
    preferred_dir_name = "out_2010_2024_10K"
    
    # Список возможных базовых путей (включая пути для Docker)
    possible_base_paths = []
    
    # 1. Через ROOT (если определен) - приоритет для Docker
    # В Docker файлы проброшены через volume mount, структура сохраняется
    if 'ROOT' in globals() and ROOT:
        root_path = Path(ROOT)
        print(f"   ROOT определен: {root_path}")
        
        # Вариант 1: EDGAR на уровень выше от ROOT (stressTest/../EDGAR)
        root_parent = root_path.parent
        edgar_via_root_1 = root_parent / "EDGAR"
        if os.path.exists(str(edgar_via_root_1)):
            possible_base_paths.append(str(edgar_via_root_1))
            print(f"   ✅ Найден EDGAR на уровень выше: {edgar_via_root_1}")
        
        # Вариант 2: EDGAR на том же уровне, что и ROOT (../EDGAR относительно stressTest)
        # Если ROOT = /workspace/stressTest, то EDGAR может быть /workspace/EDGAR
        edgar_via_root_2 = root_path / ".." / "EDGAR"
        edgar_via_root_2_resolved = edgar_via_root_2.resolve()
        if os.path.exists(str(edgar_via_root_2_resolved)) and str(edgar_via_root_2_resolved) not in possible_base_paths:
            possible_base_paths.append(str(edgar_via_root_2_resolved))
            print(f"   ✅ Найден EDGAR через resolve: {edgar_via_root_2_resolved}")
        
        # Вариант 3: EDGAR внутри ROOT (stressTest/EDGAR) - маловероятно, но проверим
        edgar_via_root_3 = root_path / "EDGAR"
        if os.path.exists(str(edgar_via_root_3)) and str(edgar_via_root_3) not in possible_base_paths:
            possible_base_paths.append(str(edgar_via_root_3))
            print(f"   ✅ Найден EDGAR внутри ROOT: {edgar_via_root_3}")
    
    # 2. Прямой путь на хосте (для локального выполнения)
    host_path = "/Users/arturhusnutdinov/Documents/IT Development/Docker/EDGAR"
    if os.path.exists(host_path) and host_path not in possible_base_paths:
        possible_base_paths.append(host_path)
    
    # 3. Через переменные окружения (для Docker)
    edgar_env = os.environ.get('EDGAR_DIR') or os.environ.get('EDGAR_PATH')
    if edgar_env and os.path.exists(edgar_env):
        if edgar_env not in possible_base_paths:
            possible_base_paths.append(edgar_env)
            print(f"   ✅ Найден путь через переменную окружения: {edgar_env}")
    
    # 4. Стандартные Docker пути (если volume смонтирован в стандартные места)
    docker_paths = [
        "/workspace/EDGAR",
        "/data/EDGAR",
        "/mnt/EDGAR",
        "/app/EDGAR",
    ]
    for docker_path in docker_paths:
        if os.path.exists(docker_path):
            if docker_path not in possible_base_paths:
                possible_base_paths.append(docker_path)
                print(f"   ✅ Найден Docker путь: {docker_path}")
    
    # 5. Через домашнюю директорию (fallback)
    home_edgar = os.path.expanduser("~/Documents/IT Development/Docker/EDGAR")
    if home_edgar not in possible_base_paths and os.path.exists(home_edgar):
        possible_base_paths.append(home_edgar)
    
    # 6. Через Path.home() (fallback)
    path_home_edgar = str(Path.home() / "Documents/IT Development/Docker/EDGAR")
    if path_home_edgar not in possible_base_paths and os.path.exists(path_home_edgar):
        possible_base_paths.append(path_home_edgar)
    
    # Ищем директорию с файлами us_steel_IS_long.csv
    edgar_dir = None
    edgar_path_str = None
    found_dirs = []  # Все найденные директории с нужными файлами
    
    print(f"🔍 Поиск директории Edgar...")
    print(f"   Приоритетная директория: {preferred_dir_name}")
    
    for base_path_str in possible_base_paths:
        # Проверяем существование базового пути
        if not os.path.exists(base_path_str) or not os.path.isdir(base_path_str):
            continue
        
        print(f"   Проверяем: {base_path_str}")
        base_path_obj = Path(base_path_str)
        
        # Ищем директории с out_* в названии
        try:
            for item in base_path_obj.iterdir():
                if item.is_dir() and item.name.startswith("out"):
                    # Проверяем наличие нужных файлов
                    is_file_path = str(item / "us_steel_IS_long.csv")
                    all_file_path = str(item / "us_steel_all_long.csv")
                    
                    is_file_exists = os.path.exists(is_file_path)
                    all_file_exists = os.path.exists(all_file_path)
                    
                    if is_file_exists and all_file_exists:
                        found_dirs.append(item)
                        print(f"      ✅ Найдена: {item.name}")
                        # Приоритет: сначала ищем preferred_dir_name
                        if item.name == preferred_dir_name:
                            edgar_dir = item
                            edgar_path_str = str(item)
                            print(f"      ⭐ Это приоритетная директория!")
                            break
        except Exception as e:
            print(f"      ⚠️ Ошибка при поиске в {base_path_str}: {e}")
            continue
        
        # Если нашли приоритетную директорию, останавливаемся
        if edgar_dir:
            break
    
    # Если не нашли приоритетную, используем первую найденную
    if edgar_dir is None and found_dirs:
        edgar_dir = found_dirs[0]
        edgar_path_str = str(found_dirs[0])
        print(f"\n⚠️ Приоритетная директория '{preferred_dir_name}' не найдена")
        print(f"   Используется: {edgar_dir.name}")
    
    # Если директория не найдена, выводим диагностическую информацию
    if edgar_dir is None or edgar_path_str is None:
        print(f"❌ Директория Edgar не найдена")
        print("\n💡 Проверенные базовые пути:")
        possible_base_paths = [
            "/Users/arturhusnutdinov/Documents/IT Development/Docker/EDGAR",
            os.path.expanduser("~/Documents/IT Development/Docker/EDGAR"),
            str(Path.home() / "Documents/IT Development/Docker/EDGAR"),
        ]
        for base_path_str in possible_base_paths:
            exists = os.path.exists(base_path_str)
            is_dir = os.path.isdir(base_path_str) if exists else False
            print(f"   • {base_path_str} - {'✅ существует' if exists else '❌ не существует'}{' (директория)' if is_dir else ''}")
        
        print("\n📋 Ищем директории с out_*:")
        for base_path_str in possible_base_paths:
            if os.path.exists(base_path_str) and os.path.isdir(base_path_str):
                base_path_obj = Path(base_path_str)
                try:
                    for item in base_path_obj.iterdir():
                        if item.is_dir() and item.name.startswith("out"):
                            is_file_path = str(item / "us_steel_IS_long.csv")
                            is_file_exists = os.path.exists(is_file_path)
                            all_file_path = str(item / "us_steel_all_long.csv")
                            all_file_exists = os.path.exists(all_file_path)
                            status = "✅" if (is_file_exists and all_file_exists) else ("⚠️ только IS" if is_file_exists else "❌")
                            print(f"   • {item.name} - IS: {status}")
                except Exception as e:
                    print(f"   ⚠️ Ошибка при проверке {base_path_str}: {e}")
        
        print("\n💡 Укажите правильный путь вручную или проверьте наличие файлов us_steel_IS_long.csv")
    else:
        print(f"\n✅ Найдена директория Edgar: {edgar_dir.name}")
        print(f"   Путь: {edgar_path_str}")
        
        # Проверяем наличие файлов
        is_long_file = edgar_dir / "us_steel_IS_long.csv"
        all_long_file = edgar_dir / "us_steel_all_long.csv"
        
        print(f"\n📋 Проверка файлов Edgar:")
        print(f"   • us_steel_IS_long.csv: {'✅ найден' if is_long_file.exists() else '❌ не найден'}")
        print(f"   • us_steel_all_long.csv: {'✅ найден' if all_long_file.exists() else '❌ не найден'}")
        
        # Проверяем маппинг в excel_loader.yaml
        yaml_path = croot / 'configs' / 'excel_loader.yaml'
        if yaml_path.exists():
            print(f"\n✅ Конфигурация маппинга найдена: {yaml_path.relative_to(ROOT)}")
            print(f"   Маппинг RAW → CANONICAL будет применен автоматически при загрузке")
        else:
            print(f"\n⚠️ Конфигурация маппинга не найдена: {yaml_path}")
        
        print(f"\n" + "="*80)
        print(f"📋 ИНСТРУКЦИИ ПО ЗАГРУЗКЕ ДАННЫХ")
        print(f"="*80)
        print(f"\n💡 ВАЖНО:")
        print(f"   • Файлы Edgar НЕ обновляются - они остаются как исходные данные из SEC")
        print(f"   • Исправления применяются через маппинг в excel_loader.yaml (уже обновлен в ШАГ 7.1)")
        print(f"   • При загрузке через load_edgar_to_data_mart.py маппинг RAW → CANONICAL применяется автоматически")
        print(f"   • Данные для 2024 года уже исправлены в БД через ШАГ 7.1")
        print(f"   • Загрузка из Edgar перезагрузит всю историю с правильным маппингом")
        
        print(f"\n📋 ПОРЯДОК ДЕЙСТВИЙ:")
        print(f"   1. ✅ Маппинг в excel_loader.yaml обновлен (ШАГ 7.1)")
        print(f"   2. ✅ Данные для 2024 года исправлены в БД (ШАГ 7.1)")
        print(f"   3. Загрузите всю историю IS из Edgar в БД:")
        print(f"      python tools/load_edgar_to_data_mart.py --company us_steel --edgar-dir '{edgar_path_str}'")
        print(f"      Это загрузит ВСЮ историю IS из Edgar в RAW БД")
        print(f"      Маппинг RAW → CANONICAL произойдет автоматически через excel_loader.yaml")
        print(f"   4. Экспортируйте данные из БД обратно в CSV истории (ШАГ 7.2)")
        print(f"      Это синхронизирует CSV файлы истории (is_history_us_steel.csv) с БД")
        print(f"   5. Проверьте данные через ШАГ 9 (Проверка исправленных данных)")
        
        print(f"\n📋 ЦЕПОЧКА ДАННЫХ:")
        print(f"   • CSV файлы в EDGAR/{edgar_dir.name}/ → исходные данные из SEC (не изменяются)")
        print(f"   • tools/load_edgar_to_data_mart.py → читает CSV → загружает в RAW БД")
        print(f"   • excel_loader.yaml → маппинг RAW → CANONICAL (автоматически)")
        print(f"   • CSV файлы истории → используются как fallback, обновляются через ШАГ 7.2")


## 🔄 ШАГ 7.4: Загрузка всей истории из Edgar в БД (ОПЦИОНАЛЬНО)

**Цель**: Загрузить всю историю IS из обновленных файлов Edgar в БД через `load_edgar_to_data_mart.py`

**Важно**: 
- Этот шаг выполняется через командную строку, а не в ноутбуке
- Используйте команду ниже для загрузки всей истории IS из Edgar в БД
- Это перезагрузит всю историю с правильными данными для 2024 года


In [ ]:
# === Инструкции по загрузке всей истории из Edgar ===

display(Markdown("### 🔄 Инструкции по загрузке всей истории из Edgar"))

print("="*80)
print("ЗАГРУЗКА ВСЕЙ ИСТОРИИ ИЗ EDGAR В БД")
print("="*80)

print("\n💡 ВАЖНО:")
print("   Этот шаг выполняется через командную строку, а не в ноутбуке")
print("   Используйте команду ниже для загрузки всей истории IS из Edgar в БД")

print("\n📋 КОМАНДА ДЛЯ ЗАГРУЗКИ:")
print("="*80)
print(f"python tools/load_edgar_to_data_mart.py --company us_steel --edgar-dir '/Users/arturhusnutdinov/Documents/IT Development/Docker/EDGAR/out_2010_2024_10K'")
print("="*80)

print("\n📋 ЧТО ДЕЛАЕТ ЭТА КОМАНДА:")
print("   1. Читает обновленные файлы Edgar (us_steel_IS_long.csv и us_steel_all_long.csv)")
print("   2. Загружает ВСЮ историю IS из Edgar в RAW БД")
print("   3. Маппинг RAW → CANONICAL происходит автоматически через excel_loader.yaml")
print("   4. Данные становятся доступными для препроцессинга и движка модели")

print("\n💡 ПРЕИМУЩЕСТВА:")
print("   ✅ Загружается вся история (не только 2024 год)")
print("   ✅ Данные проходят через правильный маппинг RAW → CANONICAL")
print("   ✅ Автоматически применяется combine_from из excel_loader.yaml")
print("   ✅ Единый источник данных (Edgar) для всей истории")

print("\n📋 СЛЕДУЮЩИЕ ШАГИ ПОСЛЕ ЗАГРУЗКИ:")
print("   1. Выполните ШАГ 7.2 для экспорта данных из БД в CSV файлы истории")
print("   2. Перезапустите препроцессинг модели")
print("   3. Проверьте отчет о прибылях и убытках через ШАГ 9")
print("   4. Убедитесь, что Revenue - Expenses = Net Income сходится")


## 💾 ШАГ 8: Сохранение исправленных данных

In [ ]:
# === Сохранение исправленного canonical формата ===

if 'corrected_canonical_2024' not in globals():
    print("❌ Исправленный canonical формат не создан. Запустите ячейку ШАГ 6.")
else:
    display(Markdown("### 💾 Сохранение исправленных данных"))
    
    print("Доступные опции сохранения:")
    print("1. Обновить файл истории (is_history_us_steel.csv)")
    print("2. Создать файл для проверки (outputs/corrected_is_2024.csv)")
    print("3. Экспортировать маппинг (outputs/is_mapping_2024.csv)")
    
    # === Опция 1: Обновление файла истории ===
    print("\n" + "="*80)
    print("Опция 1: Обновление файла истории")
    print("="*80)
    
    hist_file = croot / "history" / "is_history_us_steel.csv"
    
    if hist_file.exists():
        hist_df = pd.read_csv(hist_file)
        
        if '2024' in hist_df.columns:
            print("\nОбновление значений для 2024 года:")
            
            updates_count = 0
            updates_log = []
            
            for idx, row in hist_df.iterrows():
                metric = row['metric']
                if metric in corrected_canonical_2024:
                    old_val = pd.to_numeric(row['2024'], errors='coerce')
                    new_val = corrected_canonical_2024[metric]
                    
                    if pd.isna(old_val) or abs(old_val - new_val) > 1000:
                        hist_df.at[idx, '2024'] = new_val
                        updates_log.append({
                            'Метрика': metric,
                            'Старое значение': f"${old_val:,.0f}" if pd.notna(old_val) else "N/A",
                            'Новое значение': f"${new_val:,.0f}",
                            'Разница': f"${new_val - (old_val if pd.notna(old_val) else 0):,.0f}"
                        })
                        print(f"  {metric}: ${old_val:,.0f} -> ${new_val:,.0f}")
                        updates_count += 1
            
            if updates_count > 0:
                # Создаём резервную копию
                backup_file = hist_file.parent / f"is_history_us_steel_backup_{pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')}.csv"
                hist_df_original = pd.read_csv(hist_file)
                hist_df_original.to_csv(backup_file, index=False)
                print(f"\n✅ Создана резервная копия: {backup_file.name}")
                
                # Сохраняем обновлённый файл
                hist_df.to_csv(hist_file, index=False)
                print(f"✅ Файл истории обновлён: {updates_count} статей изменено")
                
                # Показываем таблицу изменений
                if updates_log:
                    updates_df = pd.DataFrame(updates_log)
                    display(Markdown("#### 📋 Таблица изменений:"))
                    display(updates_df)
            else:
                print("ℹ️ Изменений не требуется - все значения уже соответствуют")
        else:
            print(f"⚠️ Колонка '2024' не найдена в файле истории")
    else:
        print(f"⚠️ Файл истории не найден: {hist_file}")
    
    # === Опция 2: Создание файла для проверки ===
    print("\n" + "="*80)
    print("Опция 2: Создание файла для проверки")
    print("="*80)
    
    check_file = croot / "outputs" / "corrected_is_2024.csv"
    check_file.parent.mkdir(parents=True, exist_ok=True)
    
    check_data = []
    for metric, value in sorted(corrected_canonical_2024.items()):
        check_data.append({
            'metric': metric,
            'year': 2024,
            'value': value
        })
    
    check_df = pd.DataFrame(check_data)
    check_df.to_csv(check_file, index=False)
    
    print(f"✅ Файл для проверки создан: {check_file.relative_to(ROOT)}")
    print(f"   Всего статей: {len(check_data)}")
    
    display(check_df.head(20))
    
    # === Опция 3: Экспорт маппинга ===
    if 'mapping_df' in locals():
        print("\n" + "="*80)
        print("Опция 3: Экспорт маппинга")
        print("="*80)
        
        mapping_file = croot / "outputs" / "is_mapping_2024.csv"
        mapping_file.parent.mkdir(parents=True, exist_ok=True)
        mapping_df.to_csv(mapping_file, index=False)
        
        print(f"✅ Маппинг сохранен: {mapping_file.relative_to(ROOT)}")
    
    print("\n✅ Все операции завершены")

## ✅ ШАГ 9: Проверка исправленных данных

In [ ]:
# === Проверка исправленных данных ===

if 'corrected_canonical_2024' not in globals():
    print("❌ Исправленный canonical формат не создан. Запустите ячейку ШАГ 6.")
else:
    display(Markdown("### ✅ Проверка исправленного canonical формата"))
    
    # Проверка отчет о прибылях и убыткаха: Revenue - Expenses = Net Income
    total_assets = corrected_canonical_2024.get('total_assets', 0)
    total_liabilities = corrected_canonical_2024.get('total_liabilities', 0)
    total_equity = corrected_canonical_2024.get('total_equity', 0)
    
    liab_equity = total_liabilities + total_equity
    balance_diff = total_assets - liab_equity
    
    print("\n📊 Проверка отчет о прибылях и убыткаха:")
    print(f"  Total Revenue: ${total_assets:,.0f}")
    print(f"  Total Liabilities: ${total_liabilities:,.0f}")
    print(f"  Total Equity: ${total_equity:,.0f}")
    print(f"  Liabilities + Equity: ${liab_equity:,.0f}")
    print(f"  Разница: ${balance_diff:,.0f}")
    
    if abs(balance_diff) < 1:
        print("  ✅ Отчет о прибылях и убытках сходится!")
    else:
        print(f"  ❌ Отчет о прибылях и убытках не сходится! Разница: ${balance_diff:,.0f}")
    
    # Сравнение с официальным отчетом
    print("\n📊 Сравнение с официальным отчетом:")
    
    official_total_assets = official_reporting_2024.get('Total assets', 0)
    official_total_liab_equity = official_reporting_2024.get('Total liabilities and stockholders equity', 0)
    
    assets_diff = abs(total_assets - official_total_assets)
    liab_equity_diff = abs(liab_equity - official_total_liab_equity)
    
    print(f"  Total Revenue (офиц.): ${official_total_assets:,.0f}")
    print(f"  Total Revenue (canonical): ${total_assets:,.0f}")
    print(f"  Разница: ${assets_diff:,.0f}")
    
    print(f"  Total Liab+Equity (офиц.): ${official_total_liab_equity:,.0f}")
    print(f"  Total Liab+Equity (canonical): ${liab_equity:,.0f}")
    print(f"  Разница: ${liab_equity_diff:,.0f}")
    
    if assets_diff < 1 and liab_equity_diff < 1:
        print("  ✅ Данные полностью соответствуют официальному отчету!")
    else:
        print(f"  ⚠️ Есть расхождения с официальным отчетом")
    
    print("\n✅ Проверка завершена")

## 🔗 ШАГ 10: Анализ объединения статей при препроцессинге

**Цель**: Понять, как при препроцессинге несколько статей объединяются в одну canonical метрику

**Задача**:
1. Проанализировать, какие статьи должны суммироваться (например, `Accounts payable to related parties` + `Accounts payable and other accrued liabilities` → `accounts_payable`)
2. Проверить, как это работает сейчас в препроцессинге
3. Показать, где это настраивается (YAML или код)
4. Предложить решение для правильного объединения статей


In [ ]:
# === Анализ объединения статей при препроцессинге ===

import yaml
from pathlib import Path

display(Markdown("### 📋 Анализ: Как объединяются статьи при препроцессинге"))

# Загружаем конфигурацию маппинга
yaml_path = croot / 'configs' / 'excel_loader.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    loader_config = yaml.safe_load(f)

is_metrics = loader_config.get('history', {}).get('IS', {}).get('required_metrics', {})

# Примеры объединения статей
merging_examples = {
    'accounts_payable': {
        'description': 'Accounts payable = Accounts payable and other accrued liabilities + Accounts payable to related parties',
        'source_metrics': [
            'accounts_payable_and_other_accrued_liabilities',  # $2,601M
            'accounts_payable_to_related_parties',  # $146M (маппится на accounts_payable_related_parties)
        ],
        'target_metric': 'accounts_payable',
        'expected_sum': 2601.0 + 146.0,  # $2,747M
        'official_value': 2601.0,  # Но в официальном отчете accounts_payable = $2,601M (уже включает accrued)
    },
}

print("\n" + "="*80)
print("📊 ПРИМЕРЫ ОБЪЕДИНЕНИЯ СТАТЕЙ")
print("="*80)

for target_metric, info in merging_examples.items():
    print(f"\n🔍 {target_metric.upper()}:")
    print(f"   Описание: {info['description']}")
    print(f"   Исходные метрики:")
    for src_metric in info['source_metrics']:
        # Проверяем, есть ли эта метрика в YAML
        if src_metric in is_metrics:
            aliases = is_metrics[src_metric].get('aliases', [])
            print(f"      • {src_metric} (алиасы: {aliases})")
        else:
            # Проверяем, маппится ли она на другую метрику
            found_mapping = False
            for canonical, config in is_metrics.items():
                aliases = config.get('aliases', [])
                if src_metric in aliases or src_metric == canonical:
                    print(f"      • {src_metric} → маппится на {canonical} (алиасы: {aliases})")
                    found_mapping = True
                    break
            if not found_mapping:
                print(f"      • {src_metric} → НЕ НАЙДЕНА в маппинге")
    
    print(f"   Целевая метрика: {info['target_metric']}")
    if info['target_metric'] in is_metrics:
        target_aliases = is_metrics[info['target_metric']].get('aliases', [])
        print(f"      Алиасы: {target_aliases}")
    
    print(f"   Ожидаемая сумма: ${info['expected_sum']:,.0f}M")
    print(f"   Официальное значение: ${info['official_value']:,.0f}M")

print("\n" + "="*80)
print("💡 КАК ЭТО РАБОТАЕТ СЕЙЧАС")
print("="*80)

print("\n📋 Текущая логика маппинга (из normalize_to_canonical в data_mart.py):")
print("   1. ПЕРВЫЙ ПРИОРИТЕТ: Прямое совпадение имени метрики")
print("   2. ВТОРОЙ ПРИОРИТЕТ: Маппинг через алиасы (один к одному)")
print("   3. ТРЕТИЙ ПРИОРИТЕТ: Частичное совпадение имени")
print("   4. ЧЕТВЕРТЫЙ ПРИОРИТЕТ: Обратное частичное совпадение")
print("\n⚠️ ПРОБЛЕМА: Нет логики СУММИРОВАНИЯ нескольких метрик в одну!")

print("\n📋 Пример для accounts_payable:")
print("   • accounts_payable_and_other_accrued_liabilities → маппится на accounts_payable (через алиас)")
print("   • accounts_payable_to_related_parties → маппится на accounts_payable_related_parties (отдельная метрика)")
print("   • Результат: accounts_payable = $2,601M (только первая метрика)")
print("   • accounts_payable_related_parties = $146M (отдельно)")
print("\n   ⚠️ ПРОБЛЕМА: Для моделирования нужна ОДНА метрика accounts_payable")
print("   ✅ ЛОГИКА: AP моделируется через оборачиваемость (turnover), детализация не нужна")
print("   ✅ РЕШЕНИЕ: Объединить accounts_payable + accounts_payable_related_parties → accounts_payable")
print("   ✅ Ожидаемое значение: $2,601M + $146M = $2,747M")

print("\n" + "="*80)
print("🔧 ГДЕ ЭТО НАСТРАИВАЕТСЯ")
print("="*80)

print("\n1. 📄 excel_loader.yaml:")
print("   • Определяет алиасы для каждой метрики")
print("   • Путь: companies/us_steel/configs/excel_loader.yaml")
print("   • Структура: history.IS.required_metrics.{metric_name}.aliases")

print("\n2. 💻 data_mart.py (normalize_to_canonical):")
print("   • Применяет маппинг из YAML")
print("   • Путь: engine/database/data_mart.py")
print("   • Метод: normalize_to_canonical()")
print("   • ⚠️ НЕ поддерживает суммирование - только один к одному маппинг")

print("\n3. 🔄 preprocess.py:")
print("   • Рассчитывает метрики из истории")
print("   • Путь: engine/model/preprocess.py")
print("   • Использует _metric_series() для поиска метрик по нескольким алиасам")
print("   • ⚠️ НЕ суммирует метрики - берет первую найденную")

print("\n" + "="*80)
print("💡 РЕШЕНИЕ: КАК ДОБАВИТЬ СУММИРОВАНИЕ")
print("="*80)

print("\n📋 Вариант 1: Расширить normalize_to_canonical в data_mart.py")
print("   • Добавить секцию 'combine' в excel_loader.yaml:")
print("     combine:")
print("       accounts_payable:")
print("         - accounts_payable_and_other_accrued_liabilities")
print("         - accounts_payable_to_related_parties")
print("   • В normalize_to_canonical: если метрика в 'combine', суммировать все источники")

print("\n📋 Вариант 2: Использовать calculated метрики в preprocess.py")
print("   • Добавить расчет в _process_calculated_metrics:")
print("     accounts_payable = accounts_payable_and_other_accrued_liabilities + accounts_payable_to_related_parties")
print("   • Сохранять как calculated метрику в БД")

print("\n📋 Вариант 3: Предобработка перед normalize_to_canonical")
print("   • В load_history_csv_to_db или перед normalize_to_canonical:")
print("     • Найти все метрики из 'combine' списка")
print("     • Суммировать их")
print("     • Создать новую строку с целевой метрикой")
print("     • Затем применить normalize_to_canonical")

print("\n✅ РЕКОМЕНДАЦИЯ: Вариант 1 (расширить normalize_to_canonical)")
print("   • Наиболее гибкий")
print("   • Настраивается через YAML")
print("   • Не требует изменений в preprocess.py")


In [ ]:
# === Проверка текущего состояния: как метрики загружаются в canonical ===

display(Markdown("### 🔍 Проверка: Как метрики загружаются в canonical формат"))

# Загружаем данные из БД
mart = get_data_mart(ROOT, COMPANY)

# Загружаем RAW данные (до маппинга)
is_raw = mart.get_history_income_statement(canonical=False)

# Загружаем CANONICAL данные (после маппинга)
is_canonical = mart.get_history_income_statement(canonical=True)

print("\n" + "="*80)
print("📊 СРАВНЕНИЕ: RAW vs CANONICAL (2024)")
print("="*80)

# Проверяем ключевые метрики для accounts_payable
ap_related_metrics = [
    'accounts_payable',
    'accounts_payable_and_other_accrued_liabilities',
    'accounts_payable_to_related_parties',
    'accounts_payable_related_parties',
    'accrued_liabilities',
]

print("\n🔍 Метрики, связанные с Accounts Payable:")

comparison_data = []
for metric in ap_related_metrics:
    raw_val = None
    canonical_val = None
    
    # Проверяем RAW
    if not is_raw.empty and 'metric' in is_raw.columns:
        raw_row = is_raw[is_raw['metric'].str.lower() == metric.lower()]
        if not raw_row.empty and '2024' in raw_row.columns:
            raw_val = pd.to_numeric(raw_row.iloc[0]['2024'], errors='coerce')
    
    # Проверяем CANONICAL
    if not is_canonical.empty and 'metric' in is_canonical.columns:
        canonical_row = is_canonical[is_canonical['metric'].str.lower() == metric.lower()]
        if not canonical_row.empty and '2024' in canonical_row.columns:
            canonical_val = pd.to_numeric(canonical_row.iloc[0]['2024'], errors='coerce')
    
    if raw_val is not None or canonical_val is not None:
        comparison_data.append({
            'Метрика': metric,
            'RAW (2024)': f"${raw_val:,.0f}" if pd.notna(raw_val) else "N/A",
            'CANONICAL (2024)': f"${canonical_val:,.0f}" if pd.notna(canonical_val) else "N/A",
            'Статус': '✅ Найдена' if canonical_val is not None else '❌ Не найдена'
        })

if comparison_data:
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df.style.set_table_styles([
        {'selector': 'th', 'props': [('font-size', '10pt'), ('padding', '5px')]},
        {'selector': 'td', 'props': [('font-size', '9pt'), ('padding', '3px')]},
    ]).set_properties(**{'text-align': 'left'}))

print("\n💡 АНАЛИЗ:")
print("   • accounts_payable_and_other_accrued_liabilities → маппится на accounts_payable")
print("   • accounts_payable_to_related_parties → маппится на accounts_payable_related_parties")
print("   • Результат: ДВЕ отдельные метрики в canonical формате")
print("   • ❌ НЕТ суммирования в одну метрику accounts_payable")

print("\n📋 ОФИЦИАЛЬНЫЙ ОТЧЕТ:")
print("   • Accounts payable and other accrued liabilities: $2,601M")
print("   • Accounts payable to related parties: $146M (отдельная статья)")
print("   • Total current liabilities = $2,601M + $146M + другие статьи = $3,373M")

print("\n💡 ЛОГИКА МОДЕЛИРОВАНИЯ:")
print("   • AP моделируется через оборачиваемость (Accounts Payable Turnover = COGS / AP)")
print("   • Детализация отчетности (related parties vs regular AP) не нужна для моделирования")
print("   • Для удобства моделирования нужно ОБЪЕДИНИТЬ в одну метрику accounts_payable")
print("   • accounts_payable = accounts_payable_and_other_accrued_liabilities + accounts_payable_to_related_parties")
print("   • Ожидаемое значение: $2,601M + $146M = $2,747M")

print("\n✅ ВЫВОД:")
print("   • Текущее состояние: ДВЕ отдельные метрики (неправильно для моделирования)")
print("   • Нужно: ОДНА метрика accounts_payable = $2,747M (сумма обеих)")
print("   • accounts_payable_related_parties НЕ должна быть отдельной метрикой в canonical")
print("   • Она должна суммироваться в accounts_payable при препроцессинге")

mart.close()


In [ ]:
# === Пример: Где может понадобиться суммирование ===

display(Markdown("### 📋 Примеры статей, которые МОГУТ объединяться"))

# Примеры потенциального объединения
potential_merges = {
    'accounts_payable': {
        'description': 'ОБЪЕДИНЕНИЕ всех видов AP в одну метрику для моделирования через оборачиваемость',
        'source_metrics': [
            'accounts_payable_and_other_accrued_liabilities',  # $2,601M
            'accounts_payable_to_related_parties',  # $146M → accounts_payable_related_parties
        ],
        'target': 'accounts_payable',
        'current_status': '❌ НЕ объединяются (НУЖНО ИСПРАВИТЬ)',
        'when_needed': 'ВСЕГДА - AP моделируется через оборачиваемость, детализация не нужна',
        'expected_value': 2601.0 + 146.0,  # $2,747M
        'calculation': 'AP Turnover = COGS / accounts_payable',
    },
    'other_current_liabilities': {
        'description': 'Объединение различных прочих текущих обязательств',
        'source_metrics': [
            'accrued_liabilities',
            'taxes_payable',
            'interest_payable',
            'other_accrued_expenses',
        ],
        'target': 'other_current_liabilities',
        'current_status': '⚠️ Частично объединяются в preprocess.py',
        'when_needed': 'Когда нужно собрать все прочие обязательства в одну статью',
    },
    'other_non_current_liabilities': {
        'description': 'Объединение прочих долгосрочных обязательств',
        'source_metrics': [
            'deferred_credits',
            'other_long_term_liabilities',
            'provisions',
        ],
        'target': 'other_non_current_liabilities',
        'current_status': '⚠️ Частично объединяются в preprocess.py',
        'when_needed': 'Когда нужно собрать все прочие долгосрочные обязательства',
    },
}

print("\n" + "="*80)
print("📊 ПРИМЕРЫ ПОТЕНЦИАЛЬНОГО ОБЪЕДИНЕНИЯ")
print("="*80)

for target, info in potential_merges.items():
    print(f"\n🔍 {target.upper()}:")
    print(f"   Описание: {info['description']}")
    print(f"   Исходные метрики: {', '.join(info['source_metrics'])}")
    print(f"   Целевая метрика: {info['target']}")
    print(f"   Текущий статус: {info['current_status']}")
    print(f"   Когда нужно: {info['when_needed']}")
    if 'expected_value' in info:
        print(f"   Ожидаемое значение: ${info['expected_value']:,.0f}M")
    if 'calculation' in info:
        print(f"   Расчет моделирования: {info['calculation']}")

print("\n" + "="*80)
print("💡 ГДЕ ЭТО УЖЕ РЕАЛИЗОВАНО")
print("="*80)

print("\n📋 В preprocess.py (_process_calculated_metrics):")
print("   • other_current_assets = суммируются различные прочие активы")
print("   • other_current_liabilities = суммируются различные прочие обязательства")
print("   • other_non_current_assets = суммируются различные прочие долгосрочные активы")
print("   • other_non_current_liabilities = суммируются различные прочие долгосрочные обязательства")

print("\n📋 Логика суммирования:")
print("   • Ищет метрики по списку алиасов")
print("   • Суммирует все найденные значения")
print("   • Сохраняет как calculated метрику")

print("\n" + "="*80)
print("🔧 КАК ДОБАВИТЬ НОВОЕ ОБЪЕДИНЕНИЕ")
print("="*80)

print("\n1. 📄 В excel_loader.yaml добавить секцию 'combine_from' для accounts_payable:")
print("""
history:
  IS:
    required_metrics:
      accounts_payable:
        aliases:
          - accounts_payable_and_other_accrued_liabilities
        combine_from:  # НОВАЯ СЕКЦИЯ - метрики для суммирования
          - accounts_payable_and_other_accrued_liabilities  # $2,601M
          - accounts_payable_to_related_parties  # $146M (маппится на accounts_payable_related_parties)
        # Результат: accounts_payable = $2,601M + $146M = $2,747M
      
      # accounts_payable_related_parties НЕ должна быть в required_metrics
      # Она используется только как источник для accounts_payable
""")

print("\n2. 💻 В data_mart.py (normalize_to_canonical) добавить логику:")
print("""
# После создания canonical_df, перед заполнением значений:
for metric in required_metrics:
    metric_config = is_metrics.get(metric, {})
    combine_from = metric_config.get('combine_from', [])
    
    if combine_from:
        # Суммируем все метрики из combine_from
        combined_value = 0.0
        for source_metric in combine_from:
            # Ищем source_metric в исходных данных
            source_row = df[df['metric'].str.lower() == source_metric.lower()]
            if not source_row.empty:
                for year in years:
                    year_str = str(year)
                    val = pd.to_numeric(source_row.iloc[0].get(year_str, 0), errors='coerce')
                    if pd.notna(val):
                        combined_value += float(val)
        
        # Записываем суммированное значение
        canonical_df.loc[canonical_df['metric'] == metric, year_str] = combined_value
""")

print("\n3. ✅ Результат:")
print("   • accounts_payable будет автоматически суммироваться из двух источников")
print("   • accounts_payable = $2,601M + $146M = $2,747M")
print("   • accounts_payable_related_parties НЕ будет в canonical формате")
print("   • Настраивается через YAML без изменения кода")
print("   • Работает при каждом вызове normalize_to_canonical")
print("   • AP Turnover = COGS / accounts_payable будет рассчитываться правильно")

print("\n" + "="*80)
print("💡 ЛОГИКА ОБЪЕДИНЕНИЯ СТАТЕЙ")
print("="*80)
print("\n📋 ПРИНЦИПЫ:")
print("   1. Детализация отчетности ≠ Детализация для моделирования")
print("   2. Если метрика моделируется через агрегированный показатель (turnover, ratio),")
print("      то детализация не нужна - объединяем в одну метрику")
print("   3. Примеры объединения:")
print("      • accounts_payable: моделируется через AP Turnover = COGS / AP")
print("        → Объединяем все виды AP в одну метрику")
print("      • accounts_receivable: моделируется через AR Turnover = Revenue / AR")
print("        → Объединяем все виды AR в одну метрику (если есть детализация)")
print("      • inventory: моделируется через Inventory Turnover = COGS / Inventory")
print("        → Объединяем все виды inventory в одну метрику")
print("\n   4. Когда НЕ объединять:")
print("      • Метрики моделируются отдельно (разные драйверы)")
print("      • Метрики имеют разную динамику и требуют отдельного прогноза")
print("      • Метрики используются в разных частях модели независимо")

print("\n📋 ДЛЯ accounts_payable:")
print("   ✅ ОБЪЕДИНЯЕМ: accounts_payable + accounts_payable_related_parties")
print("   ✅ Причина: Обе моделируются через один показатель - AP Turnover")
print("   ✅ Результат: accounts_payable = $2,747M (сумма обеих)")
print("   ✅ accounts_payable_related_parties НЕ должна быть в canonical формате")


## 🤖 ШАГ 11: Автоматический анализ и предложение объединения статей

**Цель**: Автоматически проанализировать все метрики и предложить варианты объединения

**Алгоритм**:
1. Проверить все canonical метрики - какие уже заполнены и однозначно не требуют объединения
2. Проверить незаполненные метрики - что осталось
3. Проанализировать RAW данные - какие метрики есть в исходных данных
4. Предложить варианты объединения на основе:
   - Логики моделирования (turnover, ratio)
   - Схожести названий
   - Структуры отчетности


In [ ]:
# === Автоматический анализ метрик и предложение объединения ===

display(Markdown("### 🤖 Автоматический анализ: Проверка заполненных и предложение объединения"))

mart = get_data_mart(ROOT, COMPANY)

# Загружаем данные
is_raw = mart.get_history_income_statement(canonical=False)
is_canonical = mart.get_history_income_statement(canonical=True)

# Загружаем конфигурацию
yaml_path = croot / 'configs' / 'excel_loader.yaml'
with open(yaml_path, 'r', encoding='utf-8') as f:
    loader_config = yaml.safe_load(f)

is_metrics_config = loader_config.get('history', {}).get('IS', {}).get('required_metrics', {})

# Получаем список canonical метрик из БД
canonical_metrics_from_db = mart.get_canonical_metrics_from_db('IS')
if not canonical_metrics_from_db:
    # Fallback: используем из конфига
    canonical_metrics_from_db = list(is_metrics_config.keys())

year_to_check = 2024
year_str = str(year_to_check)

print("\n" + "="*80)
print("📊 ШАГ 1: АНАЛИЗ ЗАПОЛНЕННЫХ МЕТРИК")
print("="*80)

# Анализируем каждую canonical метрику
filled_metrics = []
unfilled_metrics = []
metrics_needing_merge = []

for metric in canonical_metrics_from_db:
    # Проверяем значение в canonical формате
    canonical_val = None
    if not is_canonical.empty and 'metric' in is_canonical.columns:
        metric_row = is_canonical[is_canonical['metric'].str.lower() == metric.lower()]
        if not metric_row.empty and year_str in metric_row.columns:
            canonical_val = pd.to_numeric(metric_row.iloc[0][year_str], errors='coerce')
    
    # Проверяем, есть ли метрика в RAW данных
    raw_sources = []
    if not is_raw.empty and 'metric' in is_raw.columns:
        for raw_metric in is_raw['metric'].unique():
            if pd.notna(raw_metric):
                raw_row = is_raw[is_raw['metric'] == raw_metric]
                if not raw_row.empty and year_str in raw_row.columns:
                    raw_val = pd.to_numeric(raw_row.iloc[0][year_str], errors='coerce')
                    if pd.notna(raw_val) and abs(raw_val) > 1e-6:
                        # Проверяем маппинг
                        metric_config = is_metrics_config.get(metric, {})
                        aliases = metric_config.get('aliases', [])
                        if (raw_metric.lower() == metric.lower() or 
                            raw_metric.lower() in [a.lower() for a in aliases]):
                            raw_sources.append({
                                'metric': raw_metric,
                                'value': raw_val
                            })
    
    # Классифицируем метрику
    if pd.notna(canonical_val) and abs(canonical_val) > 1e-6:
        filled_metrics.append({
            'metric': metric,
            'value': canonical_val,
            'sources': raw_sources,
            'status': '✅ Заполнена',
            'needs_merge': len(raw_sources) > 1  # Если несколько источников - возможно нужно объединить
        })
    else:
        unfilled_metrics.append({
            'metric': metric,
            'sources': raw_sources,
            'status': '❌ Не заполнена' if len(raw_sources) == 0 else '⚠️ Есть источники, но не заполнена'
        })

print(f"\n📊 Статистика:")
print(f"   Всего canonical метрик: {len(canonical_metrics_from_db)}")
print(f"   ✅ Заполненных: {len(filled_metrics)}")
print(f"   ❌ Незаполненных: {len(unfilled_metrics)}")

# Показываем заполненные метрики с несколькими источниками (возможно нуждаются в объединении)
metrics_with_multiple_sources = [m for m in filled_metrics if m['needs_merge']]
if metrics_with_multiple_sources:
    print(f"\n⚠️ Метрики с несколькими источниками (возможно нужно объединить): {len(metrics_with_multiple_sources)}")
    for m in metrics_with_multiple_sources[:10]:  # Показываем первые 10
        print(f"   • {m['metric']}: {len(m['sources'])} источников")
        for src in m['sources']:
            print(f"      - {src['metric']}: ${src['value']:,.0f}")

mart.close()


## 📊 ШАГ 12: Анализ требований к canonical формату IS

### Цель
Определить полный набор метрик IS, которые должны быть в canonical формате, на основе анализа использования в:
1. **Препроцессинге** (`engine/model/preprocess.py`)
2. **Движке модели** (`engine/model3/core.py`, `engine/model3/io.py`)
3. **Расчетных метриках** (margins, ratios, etc.)

### Ключевые принципы маппинга данных из отчетности в canonical формат:

1. **Прямой маппинг**: Статьи из официального отчета напрямую сопоставляются с canonical метриками через `excel_loader.yaml`
2. **Суммирование статей**: Несколько статей из отчета суммируются в одну canonical метрику (например, разные виды revenue → revenue)
3. **Остальное в "прочее"**: Статьи, которые не маппятся напрямую и не суммируются, должны попадать в "прочее" (other_income, other_expenses)
4. **Полный набор для движка**: Все метрики, которые использует движок модели, должны присутствовать в canonical формате

### Методология
- Анализ кода для выявления всех используемых метрик IS
- Анализ маппинга статей из официального отчета
- Определение статей для суммирования
- Определение статей для "прочего"
- Сравнение с текущим canonical форматом
- План доработки canonical формата


In [ ]:
# === Анализ использования метрик IS в коде ===

display(Markdown("### 📋 Метрики IS, используемые в препроцессинге"))

# Метрики из _process_working_capital
wc_metrics = {
    "revenue": "Для расчета DSO (Days Sales Outstanding)",
    "cogs": "Для расчета DPO (Days Payable Outstanding) и DIO (Days Inventory Outstanding)"
}

# Метрики из _process_margin_metrics
margin_metrics = {
    "revenue": "База для всех margin ratios",
    "cogs": "Для расчета cogs_ratio = cogs / revenue",
    "sgna": "Алиас: sga, для расчета sgna_ratio = sga / revenue",
    "sga": "Алиас: sgna, для расчета sgna_ratio = sga / revenue",
    "ebitda": "Для расчета ebitda_margin = ebitda / revenue",
    "ebit": "Для расчета ebit_margin = ebit / revenue",
    "ebt": "Алиас: pretax_income, для расчета tax_effective_rate = tax_expense / ebt",
    "pretax_income": "Алиас: ebt",
    "net_income": "Для расчета net_income_margin = net_income / revenue",
    "tax_expense": "Алиасы: tax, income_tax_expense, для расчета tax_effective_rate"
}

# Метрики из _process_capex_metrics
capex_metrics = {
    "revenue": "База для расчета capex_pct_revenue и depreciation_pct_revenue",
    "depreciation": "Алиас: dep, для расчета depreciation_pct_revenue = depreciation / revenue",
    "dep": "Алиас: depreciation"
}

# Метрики из _process_debt_metrics
debt_metrics = {
    "ebitda": "Для расчета net_debt_ebitda = net_debt / ebitda",
    "interest_expense": "Для расчета ebitda_interest_coverage = ebitda / interest_expense"
}

# Метрики из _process_lease_metrics (если есть)
lease_metrics = {
    "lease_interest": "Алиас: lease_interest_expense",
    "lease_expense_operating": "Алиас: operating_lease_expense",
    "dep_rou": "Алиас: depreciation_rou, rou_depreciation",
    "depreciation_rou": "Алиас: dep_rou, rou_depreciation",
    "rou_depreciation": "Алиас: dep_rou, depreciation_rou",
    "dep_finance_leases": "Алиас: depreciation_finance_leases"
}

# Метрики из _process_revenue_forecast
revenue_forecast_metrics = {
    "revenue": "Основная метрика для прогноза через регрессию с макрофакторами"
}

# Объединяем все метрики
all_preprocess_metrics = {
    **wc_metrics,
    **margin_metrics,
    **capex_metrics,
    **debt_metrics,
    **lease_metrics,
    **revenue_forecast_metrics
}

print("="*80)
print("МЕТРИКИ IS, ИСПОЛЬЗУЕМЫЕ В ПРЕПРОЦЕССИНГЕ")
print("="*80)
print(f"\nВсего уникальных метрик: {len(set(all_preprocess_metrics.keys()))}")
print("\n📊 По категориям:")
print(f"  - Working Capital: {len(wc_metrics)} метрик")
print(f"  - Margins: {len(margin_metrics)} метрик")
print(f"  - Capex: {len(capex_metrics)} метрик")
print(f"  - Debt: {len(debt_metrics)} метрик")
print(f"  - Leases: {len(lease_metrics)} метрик")
print(f"  - Revenue Forecast: {len(revenue_forecast_metrics)} метрик")

print("\n📋 Список всех метрик:")
for metric, description in sorted(set(all_preprocess_metrics.items())):
    print(f"  • {metric:30s} - {description}")

# Создаем DataFrame для отображения
import pandas as pd
preprocess_df = pd.DataFrame([
    {"Категория": "Working Capital", "Метрика": metric, "Описание": desc}
    for metric, desc in wc_metrics.items()
] + [
    {"Категория": "Margins", "Метрика": metric, "Описание": desc}
    for metric, desc in margin_metrics.items()
] + [
    {"Категория": "Capex", "Метрика": metric, "Описание": desc}
    for metric, desc in capex_metrics.items()
] + [
    {"Категория": "Debt", "Метрика": metric, "Описание": desc}
    for metric, desc in debt_metrics.items()
] + [
    {"Категория": "Leases", "Метрика": metric, "Описание": desc}
    for metric, desc in lease_metrics.items()
] + [
    {"Категория": "Revenue Forecast", "Метрика": metric, "Описание": desc}
    for metric, desc in revenue_forecast_metrics.items()
])

display(preprocess_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px')]}
]).set_properties(**{'font-size': '9px'}))


In [ ]:
# === Анализ использования метрик IS в движке модели ===

display(Markdown("### 📋 Метрики IS, используемые в движке модели (io.py, core.py)"))

# Метрики из load_inputs (io.py)
io_metrics = {
    "revenue": "Загружается из прогноза IS для Drivers.revenue (используется в моделировании)"
}

# Метрики, которые могут использоваться в HistoricState (но обычно из BS)
# HistoricState не содержит метрик IS напрямую, но они используются для расчета Drivers

# Метрики из ThreeStatementModel (core.py)
# В core.py IS метрики не загружаются напрямую из истории,
# но используются через Drivers (которые рассчитываются на основе истории IS)

print("="*80)
print("МЕТРИКИ IS, ИСПОЛЬЗУЕМЫЕ В ДВИЖКЕ МОДЕЛИ")
print("="*80)
print("\n📋 Основные метрики:")
print("  • revenue - основная метрика для прогноза и моделирования")
print("    - Используется в Drivers.revenue")
print("    - Используется для расчета различных ratios (capex_pct_revenue, etc.)")
print("    - Используется для расчета debt_target_pct_revenue")
print("    - Используется для расчета ppe_net_pct_revenue")
print("    - Используется для расчета intang_pct_revenue")

print("\n💡 Важно:")
print("  - Движок модели НЕ загружает IS метрики напрямую из истории")
print("  - Вместо этого используются Drivers, которые рассчитываются на основе:")
print("    1. Истории IS (для расчета ratios и margins)")
print("    2. Прогноза IS (для revenue)")
print("  - Все остальные IS метрики моделируются через ratios и margins")

# Создаем DataFrame
io_df = pd.DataFrame([
    {"Компонент": "io.py (load_inputs)", "Метрика": "revenue", "Использование": "Загрузка прогноза revenue в Drivers"},
    {"Компонент": "core.py (ThreeStatementModel)", "Метрика": "revenue (через Drivers)", "Использование": "Моделирование через Drivers.revenue"},
    {"Компонент": "core.py (ThreeStatementModel)", "Метрика": "cogs (через Drivers)", "Использование": "Моделирование через Drivers.cogs_ratio"},
    {"Компонент": "core.py (ThreeStatementModel)", "Метрика": "sga (через Drivers)", "Использование": "Моделирование через Drivers.sgna_ratio"},
    {"Компонент": "core.py (ThreeStatementModel)", "Метрика": "ebitda, ebit, net_income", "Использование": "Расчет через revenue, cogs, sga, depreciation"},
])

display(io_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px')]}
]).set_properties(**{'font-size': '9px'}))


In [ ]:
# === Сравнение текущего canonical формата с требованиями ===

display(Markdown("### 🔍 Сравнение текущего canonical формата с требованиями"))

# Загружаем текущий canonical формат из БД
if 'canonical_2024' in globals() and canonical_2024:
    current_canonical_metrics = set(canonical_2024.keys())
else:
    # Загружаем из БД
    if 'mart' not in globals():
        from engine.database.data_mart import get_data_mart
        mart = get_data_mart(ROOT, COMPANY)
    
    canonical_is = mart.get_history_income_statement(canonical=True)
    if not canonical_is.empty:
        current_canonical_metrics = set(canonical_is['metric'].unique())
    else:
        current_canonical_metrics = set()

# Определяем требуемые метрики (уникальные из всех источников)
required_metrics_set = set(all_preprocess_metrics.keys())

# Метрики, которые должны быть в canonical формате (основные, без алиасов)
core_canonical_metrics = {
    "revenue",           # Основная метрика
    "cogs",              # Основная метрика
    "gross_profit",      # Может быть рассчитана, но лучше иметь
    "sga",               # Основная метрика (алиас: sgna)
    "rnd",               # Может быть 0, но должна быть
    "ebitda",            # Основная метрика
    "depreciation_owned", # Основная метрика (алиас: depreciation, dep)
    "depreciation_rou",   # Основная метрика (алиас: dep_rou, depreciation_rou, rou_depreciation)
    "amortization",      # Основная метрика
    "total_da",          # Может быть рассчитана, но лучше иметь
    "gain_loss_on_disposal", # Основная метрика
    "ebit",              # Основная метрика
    "interest_income",   # Основная метрика
    "interest_expense",  # Основная метрика
    "lease_interest",    # Основная метрика (алиас: lease_interest_expense)
    "ebt",               # Может быть рассчитана, но лучше иметь (алиас: pretax_income)
    "tax_expense",       # Основная метрика (алиас: tax, income_tax_expense)
    "net_income",        # Основная метрика
    "eps_basic",         # Может быть рассчитана, но лучше иметь
    "eps_diluted",       # Может быть рассчитана, но лучше иметь
}

# Дополнительные метрики для лизинга (если используются)
lease_canonical_metrics = {
    "lease_expense_operating",  # Алиас: operating_lease_expense
    "dep_finance_leases",       # Алиас: depreciation_finance_leases
}

# Объединяем все требуемые метрики
all_required_canonical = core_canonical_metrics | lease_canonical_metrics

print("="*80)
print("СРАВНЕНИЕ ТЕКУЩЕГО CANONICAL ФОРМАТА С ТРЕБОВАНИЯМИ")
print("="*80)

print(f"\n📊 Статистика:")
print(f"  - Текущих метрик в canonical: {len(current_canonical_metrics)}")
print(f"  - Требуемых метрик (core): {len(core_canonical_metrics)}")
print(f"  - Требуемых метрик (с лизингом): {len(all_required_canonical)}")

# Находим недостающие метрики
missing_metrics = all_required_canonical - current_canonical_metrics
extra_metrics = current_canonical_metrics - all_required_canonical

print(f"\n✅ Найдено метрик: {len(current_canonical_metrics & all_required_canonical)}")
print(f"❌ Недостает метрик: {len(missing_metrics)}")
print(f"⚠️ Лишних метрик: {len(extra_metrics)}")

if missing_metrics:
    print(f"\n❌ НЕДОСТАЮЩИЕ МЕТРИКИ ({len(missing_metrics)}):")
    for metric in sorted(missing_metrics):
        # Определяем категорию
        category = "Core"
        if metric in lease_canonical_metrics:
            category = "Leases"
        elif metric in {"gross_profit", "total_da", "ebt"}:
            category = "Calculated (but should exist)"
        print(f"  • {metric:30s} [{category}]")

if extra_metrics:
    print(f"\n⚠️ ЛИШНИЕ МЕТРИКИ ({len(extra_metrics)}):")
    for metric in sorted(extra_metrics):
        print(f"  • {metric}")

# Создаем таблицу сравнения
comparison_data = []
for metric in sorted(all_required_canonical):
    in_current = metric in current_canonical_metrics
    status = "✅" if in_current else "❌"
    comparison_data.append({
        "Метрика": metric,
        "Требуется": "Да",
        "В текущем canonical": "Да" if in_current else "Нет",
        "Статус": status
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px')]}
]).set_properties(**{'font-size': '9px'}).apply(
    lambda x: ['background-color: #ffcccc' if x['Статус'] == '❌' else '' for _ in x], axis=1
))


In [ ]:
# === Анализ маппинга статей из официального отчета ===

display(Markdown("### 🔗 Анализ маппинга статей из официального отчета в canonical формат"))

print("="*80)
print("АНАЛИЗ МАППИНГА СТАТЕЙ ИЗ ОФИЦИАЛЬНОГО ОТЧЕТА")
print("="*80)

# Загружаем данные из официального отчета (если доступны)
if 'official_reporting_2024' in globals() and official_reporting_2024:
    official_items = list(official_reporting_2024.keys())
    print(f"\n📋 Статей в официальном отчете: {len(official_items)}")
    
    # Анализируем маппинг
    mapping_analysis = {
        "Прямой маппинг": [],
        "Суммирование": [],
        "В прочее": [],
        "Не определено": []
    }
    
    # Проверяем маппинг через official_to_canonical_map и correct_mapping_dict
    for item in official_items:
        mapped = False
        
        # Проверяем прямой маппинг
        if 'official_to_canonical_map' in globals() and item in official_to_canonical_map:
            canonical = official_to_canonical_map[item]
            mapping_analysis["Прямой маппинг"].append({
                "Официальная статья": item,
                "Canonical метрика": canonical,
                "Значение": official_reporting_2024[item]
            })
            mapped = True
        
        # Проверяем correct_mapping_dict
        if not mapped and 'correct_mapping_dict' in globals() and correct_mapping_dict and item in correct_mapping_dict:
            canonical = correct_mapping_dict[item]
            mapping_analysis["Прямой маппинг"].append({
                "Официальная статья": item,
                "Canonical метрика": canonical,
                "Значение": official_reporting_2024[item]
            })
            mapped = True
        
        # Проверяем, должна ли статья суммироваться
        # Например, "Net sales" + "Net sales to related parties" → "revenue"
        if not mapped:
            if "sales" in item.lower() or "revenue" in item.lower():
                mapping_analysis["Суммирование"].append({
                "Официальная статья": item,
                "Canonical метрика": "revenue (суммируется)",
                "Значение": official_reporting_2024[item]
            })
                mapped = True
        
        # Если не маппится и не суммируется, должна попасть в "прочее"
        if not mapped:
            # Определяем, это доход или расход
            value = official_reporting_2024[item]
            is_income = False
            is_expense = False
            
            # Простая эвристика: если значение положительное и содержит "income", "gain", "revenue" → доход
            if any(keyword in item.lower() for keyword in ["income", "gain", "revenue", "sales"]):
                is_income = True
            # Если значение отрицательное или содержит "expense", "loss", "charge", "cost" → расход
            elif any(keyword in item.lower() for keyword in ["expense", "loss", "charge", "cost", "impairment"]):
                is_expense = True
            
            if is_income:
                mapping_analysis["В прочее"].append({
                    "Официальная статья": item,
                    "Canonical метрика": "other_income (требуется добавить)",
                    "Значение": official_reporting_2024[item],
                    "Тип": "Доход"
                })
            elif is_expense:
                mapping_analysis["В прочее"].append({
                    "Официальная статья": item,
                    "Canonical метрика": "other_expenses (требуется добавить)",
                    "Значение": official_reporting_2024[item],
                    "Тип": "Расход"
                })
            else:
                mapping_analysis["Не определено"].append({
                    "Официальная статья": item,
                    "Значение": official_reporting_2024[item]
                })
    
    print(f"\n📊 Результаты анализа маппинга:")
    print(f"  ✅ Прямой маппинг: {len(mapping_analysis['Прямой маппинг'])} статей")
    print(f"  🔗 Суммирование: {len(mapping_analysis['Суммирование'])} статей")
    print(f"  📦 В прочее: {len(mapping_analysis['В прочее'])} статей")
    print(f"  ❓ Не определено: {len(mapping_analysis['Не определено'])} статей")
    
    # Показываем детали
    if mapping_analysis["В прочее"]:
        print(f"\n📦 СТАТЬИ, КОТОРЫЕ ДОЛЖНЫ ПОПАСТЬ В 'ПРОЧЕЕ':")
        other_df = pd.DataFrame(mapping_analysis["В прочее"])
        display(other_df.style.set_table_styles([
            {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
            {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px')]}
        ]).set_properties(**{'font-size': '9px'}))
        
        print("\n💡 Рекомендации:")
        print("  - Статьи типа 'доход' должны суммироваться в 'other_income'")
        print("  - Статьи типа 'расход' должны суммироваться в 'other_expenses'")
        print("  - Эти метрики должны быть добавлены в canonical формат IS")
else:
    print("\n⚠️ Данные из официального отчета не загружены. Запустите ШАГ 1.")

# === План доработки canonical формата IS ===

display(Markdown("### 📋 План доработки canonical формата IS"))

print("="*80)
print("ПЛАН ДОРАБОТКИ CANONICAL ФОРМАТА IS")
print("="*80)

print("\n🎯 ЦЕЛЬ:")
print("  Обеспечить полный набор метрик IS в canonical формате для:")
print("  1. ✅ Прямого маппинга статей из официального отчета")
print("  2. ✅ Суммирования нескольких статей в одну canonical метрику")
print("  3. ✅ Попадания остальных статей в 'прочее' (other_income, other_expenses)")
print("  4. ✅ Корректной работы препроцессинга")
print("  5. ✅ Корректной работы движка модели")
print("  6. ✅ Правильного расчета всех ratios и margins")

print("\n📋 ЭТАПЫ ДОРАБОТКИ:")

print("\n0️⃣ ПРОВЕРКА МАППИНГА ИЗ ОФИЦИАЛЬНОГО ОТЧЕТА:")
print("  ✅ Все статьи из официального отчета должны быть сопоставлены:")
print("     - Прямой маппинг: через official_to_canonical_map и excel_loader.yaml")
print("     - Суммирование: несколько статей → одна canonical метрика (через combine_from в YAML)")
print("     - В прочее: остальные статьи → other_income или other_expenses")
print("  ✅ Проверить, что все статьи учтены (нет 'Не определено')")
print("  ✅ Добавить метрики other_income и other_expenses в canonical формат (если нужны)")

print("\n1️⃣ ПРОВЕРКА ОСНОВНЫХ МЕТРИК:")
print("  ✅ revenue - должна быть")
print("  ✅ cogs - должна быть")
print("  ✅ sga - должна быть (или sgna как алиас)")
print("  ✅ ebitda - должна быть")
print("  ✅ ebit - должна быть")
print("  ✅ ebt - должна быть (или рассчитываться)")
print("  ✅ net_income - должна быть")
print("  ✅ tax_expense - должна быть")

print("\n2️⃣ ПРОВЕРКА МЕТРИК DEPRECIATION/AMORTIZATION:")
print("  ✅ depreciation_owned - должна быть (или depreciation как алиас)")
print("  ✅ depreciation_rou - должна быть (или dep_rou как алиас)")
print("  ✅ amortization - должна быть")
print("  ✅ total_da - должна быть (или рассчитываться)")

print("\n3️⃣ ПРОВЕРКА МЕТРИК INTEREST:")
print("  ✅ interest_income - должна быть")
print("  ✅ interest_expense - должна быть")
print("  ✅ lease_interest - должна быть (если есть лизинг)")

print("\n4️⃣ ПРОВЕРКА ДОПОЛНИТЕЛЬНЫХ МЕТРИК:")
print("  ✅ gross_profit - должна быть (или рассчитываться: revenue - cogs)")
print("  ✅ gain_loss_on_disposal - должна быть")
print("  ✅ eps_basic - должна быть (если доступна)")
print("  ✅ eps_diluted - должна быть (если доступна)")

print("\n5️⃣ ПРОВЕРКА МЕТРИК ЛИЗИНГА (если используется):")
print("  ⚠️ lease_expense_operating - должна быть (если есть operating leases)")
print("  ⚠️ dep_finance_leases - должна быть (если есть finance leases)")

print("\n6️⃣ ПРОВЕРКА АЛИАСОВ И СУММИРОВАНИЯ В excel_loader.yaml:")
print("  ✅ Убедиться, что все алиасы правильно настроены:")
print("     - sgna → sga")
print("     - depreciation, dep → depreciation_owned")
print("     - dep_rou, depreciation_rou, rou_depreciation → depreciation_rou")
print("     - tax, income_tax_expense → tax_expense")
print("     - ebt, pretax_income → ebt")
print("     - lease_interest_expense → lease_interest")
print("  ✅ Убедиться, что суммирование статей настроено через combine_from:")
print("     - Пример: 'Net sales' + 'Net sales to related parties' → revenue")
print("     - Пример: 'Other losses (gains), net' + 'Restructuring charges' → other_expenses")
print("     - Пример: 'Earnings from investees' + 'Gain on equity investee transactions' → other_income")

print("\n7️⃣ ПРОВЕРКА РАСЧЕТНЫХ МЕТРИК:")
print("  ✅ Если метрика может быть рассчитана, но отсутствует в данных:")
print("     - gross_profit = revenue - cogs")
print("     - total_da = depreciation_owned + depreciation_rou + amortization")
print("     - ebt = ebit + interest_income - interest_expense - lease_interest")
print("  ✅ Эти метрики должны быть добавлены в canonical формат")

print("\n8️⃣ ПРОВЕРКА ЗНАКОВ:")
print("  ✅ Убедиться, что знаки метрик соответствуют конвенции:")
print("     - revenue: положительный")
print("     - cogs: отрицательный (или положительный, но вычитается)")
print("     - expenses: отрицательный (или положительный, но вычитается)")
print("     - interest_income: положительный")
print("     - interest_expense: отрицательный (или положительный, но вычитается)")
print("     - tax_expense: отрицательный (или положительный, но вычитается)")

print("\n9️⃣ ПРОВЕРКА СООТВЕТСТВИЯ С ОФИЦИАЛЬНЫМ ОТЧЕТОМ:")
print("  ✅ Все метрики из официального отчета должны быть сопоставлены:")
print("     - Прямой маппинг: значения должны совпадать (с учетом знаков)")
print("     - Суммирование: сумма статей должна совпадать с canonical метрикой")
print("     - В прочее: сумма статей должна совпадать с other_income/other_expenses")
print("  ✅ Проверка баланса: Revenue - Expenses = Net Income")
print("  ✅ Отсутствующие метрики должны быть добавлены или объяснены")

print("\n🔟 ФИНАЛЬНАЯ ПРОВЕРКА:")
print("  ✅ Все метрики из required_metrics в excel_loader.yaml присутствуют")
print("  ✅ Все метрики используются в препроцессинге")
print("  ✅ Все метрики могут быть загружены в движок модели")
print("  ✅ Баланс IS сходится: Revenue - Expenses = Net Income")

# Создаем чеклист
checklist_data = [
    {"Этап": "0. Маппинг из отчета", "Метрики": "Прямой маппинг, суммирование, прочее", "Статус": "Проверить"},
    {"Этап": "1. Основные метрики", "Метрики": "revenue, cogs, sga, ebitda, ebit, ebt, net_income, tax_expense", "Статус": "Проверить"},
    {"Этап": "2. Depreciation/Amortization", "Метрики": "depreciation_owned, depreciation_rou, amortization, total_da", "Статус": "Проверить"},
    {"Этап": "3. Interest", "Метрики": "interest_income, interest_expense, lease_interest", "Статус": "Проверить"},
    {"Этап": "4. Дополнительные", "Метрики": "gross_profit, gain_loss_on_disposal, eps_basic, eps_diluted", "Статус": "Проверить"},
    {"Этап": "5. Прочее", "Метрики": "other_income, other_expenses (если нужны)", "Статус": "Добавить"},
    {"Этап": "6. Лизинг", "Метрики": "lease_expense_operating, dep_finance_leases", "Статус": "Опционально"},
    {"Этап": "7. Алиасы и суммирование в YAML", "Метрики": "Все алиасы и combine_from настроены", "Статус": "Проверить"},
    {"Этап": "8. Расчетные метрики", "Метрики": "gross_profit, total_da, ebt (если отсутствуют)", "Статус": "Добавить"},
    {"Этап": "9. Знаки", "Метрики": "Все метрики с правильными знаками", "Статус": "Проверить"},
    {"Этап": "10. Соответствие отчету", "Метрики": "Все метрики из официального отчета", "Статус": "Проверить"},
    {"Этап": "11. Финальная проверка", "Метрики": "Все метрики присутствуют и используются", "Статус": "Проверить"},
]

checklist_df = pd.DataFrame(checklist_data)
display(checklist_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px')]}
]).set_properties(**{'font-size': '9px'}))


## 🔧 ШАГ 13: Дополнение БД недостающими метриками

### Цель
Дополнить таблицу `canonical_metrics` в БД всеми необходимыми метриками IS без переименования существующих.

### Принципы
- ✅ Названия метрик соответствуют snake_case конвенции (стандарт для БД/кода)
- ✅ Названия понятны и соответствуют финансовой терминологии
- ✅ НЕ переименовываем существующие метрики (чтобы не сломать историю)
- ✅ Дополняем БД только недостающими метриками


In [ ]:
# === Анализ соответствия названий метрик общей практике и дополнение БД ===

display(Markdown("### 📊 Анализ названий метрик и дополнение БД"))

print("="*80)
print("АНАЛИЗ СООТВЕТСТВИЯ НАЗВАНИЙ МЕТРИК ОБЩЕЙ ПРАКТИКЕ")
print("="*80)

# Загружаем текущие метрики из БД
if 'mart' not in globals():
    from engine.database.data_mart import get_data_mart
    mart = get_data_mart(ROOT, COMPANY)

canonical_metrics_is_db = set(mart.get_canonical_metrics_from_db("IS"))

# Определяем требуемые метрики на основе анализа кода и практики
required_metrics_is = {
    "revenue", "cogs", "gross_profit", "sga", "rnd",
    "ebitda", "depreciation_owned", "depreciation_rou", "amortization", "total_da",
    "gain_loss_on_disposal", "ebit",
    "interest_income", "interest_expense", "lease_interest",
    "other_income", "other_expenses",  # Новые метрики для "прочего"
    "ebt", "tax_expense", "net_income",
    "eps_basic", "eps_diluted"
}

print("\n📊 Текущие метрики в БД:")
for i, metric in enumerate(sorted(canonical_metrics_is_db), 1):
    print(f"  {i:2d}. {metric}")

print(f"\n📋 Всего метрик в БД: {len(canonical_metrics_is_db)}")
print(f"📋 Требуемых метрик: {len(required_metrics_is)}")

# Находим недостающие метрики
missing_metrics = required_metrics_is - canonical_metrics_is_db

print(f"\n❌ Отсутствует в БД: {len(missing_metrics)} метрик")
if missing_metrics:
    for metric in sorted(missing_metrics):
        print(f"   • {metric}")

print("\n💡 Анализ названий:")
print("  ✅ Названия соответствуют snake_case конвенции (стандарт для БД/кода)")
print("  ✅ Названия понятны и соответствуют финансовой терминологии")
print("  ✅ Используются стандартные аббревиатуры (cogs, sga, ebit, ebt, eps)")
print("  ✅ Нет необходимости переименовывать существующие метрики")

# Дополняем БД недостающими метриками
if missing_metrics:
    print(f"\n🔧 Дополнение БД недостающими метриками ({len(missing_metrics)}):")
    
    cursor = mart.db.conn.cursor()
    
    # Определяем порядок для новых метрик (вставляем после существующих)
    max_order = 0
    cursor.execute("""
        SELECT MAX(metric_order) FROM canonical_metrics 
        WHERE model_type = ? AND statement_type = ?
    """, (mart.get_model_type(), "IS"))
    result = cursor.fetchone()
    if result and result[0]:
        max_order = result[0]
    
    # Описания для новых метрик
    descriptions = {
        "other_income": "Прочие доходы (сумма статей, не попавших в основные метрики)",
        "other_expenses": "Прочие расходы (сумма статей, не попавших в основные метрики)"
    }
    
    added_count = 0
    for i, metric in enumerate(sorted(missing_metrics), 1):
        order = max_order + i
        description = descriptions.get(metric, "")
        
        try:
            cursor.execute("""
                INSERT OR IGNORE INTO canonical_metrics 
                (model_type, statement_type, metric_name, metric_order, description, is_required, default_value)
                VALUES (?, ?, ?, ?, ?, ?, ?)
            """, (mart.get_model_type(), "IS", metric, order, description, True, 0.0))
            
            if cursor.rowcount > 0:
                print(f"   ✅ Добавлено: {metric} (порядок: {order})")
                added_count += 1
            else:
                print(f"   ℹ️ Уже существует: {metric}")
        except Exception as e:
            print(f"   ❌ Ошибка при добавлении {metric}: {e}")
    
    mart.db.conn.commit()
    
    print(f"\n✅ БД дополнена: добавлено {added_count} новых метрик")
    
    # Проверяем результат
    canonical_metrics_is_db_updated = set(mart.get_canonical_metrics_from_db("IS"))
    print(f"\n📊 Обновленное количество метрик в БД: {len(canonical_metrics_is_db_updated)}")
    
    if len(canonical_metrics_is_db_updated) >= len(required_metrics_is):
        print("✅ Все требуемые метрики присутствуют в БД")
    else:
        still_missing = required_metrics_is - canonical_metrics_is_db_updated
        print(f"⚠️ Все еще отсутствует: {len(still_missing)} метрик")
        for metric in sorted(still_missing):
            print(f"   • {metric}")
else:
    print("\n✅ Все требуемые метрики уже присутствуют в БД")

# Создаем таблицу сравнения
comparison_data = []
for metric in sorted(required_metrics_is):
    in_db = metric in canonical_metrics_is_db
    status = "✅" if in_db else "❌"
    comparison_data.append({
        "Метрика": metric,
        "В БД": "Да" if in_db else "Нет",
        "Статус": status
    })

comparison_df = pd.DataFrame(comparison_data)
display(comparison_df.style.set_table_styles([
    {'selector': 'th', 'props': [('font-size', '10px'), ('padding', '5px'), ('text-align', 'left')]},
    {'selector': 'td', 'props': [('font-size', '9px'), ('padding', '3px')]}
]).set_properties(**{'font-size': '9px'}).apply(
    lambda x: ['background-color: #ffcccc' if x['Статус'] == '❌' else '' for _ in x], axis=1
))
